<a href="https://colab.research.google.com/github/erdemustun/Analizler/blob/main/Yapi_tarayici_V3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Formasyon – BIST Tarayıcı (Colab)
Tüm BIST hisselerini verilen periyotta tarar; **çalıştırdığınız andaki en son durumu** tek bir Excel dosyasında verir. Her yapı türü için ayrı sütun çifti (▲/▼ + tarih, kaç bar önce) ve en sonda son barda yukarı / aşağı kırılanlar yer alır.

**Sıra:** 1 kurulum → 2 motor → 3 grafik → 4 ayarlar → 5 hisse listesi → 6 tarama fonksiyonları → 7 **taramayı çalıştır (Excel iner)** → 8 hisse detayı (açık renkli etkileşimli grafik).

*Yatırım tavsiyesi değildir.*

## 1 · Kurulum

In [1]:
!pip install -q tvdatafeed-enhanced==2.2.1 tradingview-screener plotly pandas openpyxl numpy tqdm

from google.colab import files

## 2 · Motor

In [2]:
# Formasyon Tarama kodu


import argparse
import math
import os
import sys
from dataclasses import dataclass

import numpy as np
import pandas as pd

ARALIKLAR = {
    "1m": "in_1_minute", "3m": "in_3_minute", "5m": "in_5_minute", "15m": "in_15_minute",
    "30m": "in_30_minute", "45m": "in_45_minute", "1h": "in_1_hour", "2h": "in_2_hour",
    "3h": "in_3_hour", "4h": "in_4_hour", "1D": "in_daily", "1W": "in_weekly", "1M": "in_monthly",
}


# ═══════════════════════ AYARLAR (Pine varsayılanları) ═══════════════════════
@dataclass
class Ayar:
    # 1 · Genel
    swing: int = 5
    # 2 · Piyasa yapısı
    bos_kapanis: bool = True          # True: kapanışla kırılım, False: fitille
    yapi_max: int = 100
    # 3 · Formasyonlar
    fm_acik: bool = True
    fm_max: int = 10
    fm_swing: int = 0                 # 0 = yapı ile aynı
    fm_hs: bool = True
    fm_tpl: bool = True
    fm_dbl: bool = True
    fm_cup: bool = True
    fm_tri: bool = True
    fm_wdg: bool = True
    fm_rect: bool = True
    fm_chan: bool = True
    fm_flag: bool = True
    fm_broad: bool = False
    fm_tol: float = 15.0              # seviye toleransı (% boy)
    fm_bas: float = 10.0              # baş farkı (% boy)
    fm_min_atr: float = 1.5
    fm_sym: float = 2.5
    fm_trend: bool = True
    fm_dokunus: int = 5               # 4 veya 5
    fm_yatay: float = 0.25
    fm_direk: float = 2.0
    fm_bekle: int = 60
    fm_hedef: bool = True
    fm_gt: bool = True                # geri test
    fm_gt_bar: int = 20
    fm_gt_tol: float = 0.25
    fm_hacim: bool = True
    fm_hacim_len: int = 20
    fm_hacim_kat: float = 1.2
    fm_stat: bool = True
    # 5 · Wyckoff
    wy_acik: bool = True
    wy_bir: bool = True
    wy_dag: bool = True
    wy_trend_len: int = 50
    wy_clx_vol: float = 2.0
    wy_clx_rng: float = 1.5
    wy_min_h: float = 2.0
    wy_max_h: float = 8.0
    wy_min_b: int = 10
    wy_ar_max: int = 30
    wy_max_bar: int = 150
    wy_spr_dep: float = 0.50
    wy_spr_bar: int = 3
    wy_spr_sart: bool = True
    wy_max: int = 5
    wy_olc: bool = True
    wy_ema: int = 200


# ═══════════════════════ YARDIMCILAR ═══════════════════════
isnan = math.isnan


def fp(p):
    """Fiyat biçimi."""
    if p is None or (isinstance(p, float) and isnan(p)):
        return "—"
    a = abs(p)
    d = 2 if a >= 100 else 4 if a >= 1 else 6
    return f"{p:.{d}f}"


def yuzde(a, b):
    return "—" if b == 0 else "%" + str(int(100.0 * a / b + 0.5))


def Y(b1, p1, b2, p2, x):
    """İki noktadan geçen doğrunun x'teki değeri."""
    return p1 + (p2 - p1) * float(x - b1) / float(b2 - b1)


def pivot_high(h, L):
    n = len(h)
    out = np.full(n, np.nan)
    for i in range(2 * L, n):
        c = i - L
        if h[c] >= h[c - L:i + 1].max():
            out[i] = h[c]
    return out


def pivot_low(l, L):
    n = len(l)
    out = np.full(n, np.nan)
    for i in range(2 * L, n):
        c = i - L
        if l[c] <= l[c - L:i + 1].min():
            out[i] = l[c]
    return out


def atr_rma(h, l, c, n=14):
    tr = np.empty(len(h))
    tr[0] = h[0] - l[0]
    for i in range(1, len(h)):
        tr[i] = max(h[i] - l[i], abs(h[i] - c[i - 1]), abs(l[i] - c[i - 1]))
    out = np.full(len(h), np.nan)
    if len(h) >= n:
        out[n - 1] = tr[:n].mean()
        for i in range(n, len(h)):
            out[i] = (out[i - 1] * (n - 1) + tr[i]) / n
    return out


# ═══════════════════════ DURUM SINIFLARI ═══════════════════════
@dataclass
class Rev:                      # boyun kırılımı bekleyen dönüş formasyonu
    act: bool = False
    b1: int = 0
    p1: float = 0.0
    b2: int = 1
    p2: float = 0.0
    st: int = 0
    hgt: float = 0.0
    inv: float = 0.0
    nm: str = ""
    rec: dict = None


@dataclass
class Lin:                      # kırılım bekleyen çizgi formasyonu
    act: bool = False
    uB1: int = 0
    uP1: float = 0.0
    uB2: int = 0
    uP2: float = 0.0
    dB1: int = 0
    dP1: float = 0.0
    dB2: int = 0
    dP2: float = 0.0
    st: int = 0
    ptEnd: int = 0
    apex: int = None
    dir: int = 0
    hgt: float = 0.0
    pole: float = 0.0
    nm: str = ""
    rec: dict = None


class Wy:                       # Wyckoff aralığı (dağıtım ters çevrilmiş fiyatla birikim gibi işlenir)
    def __init__(self, s):
        self.s = s
        self.st = 0             # 0 boş · 1 Faz A · 2 Faz B · 3 Faz C · 4 Faz D
        self.scM = self.hiM = self.arM = self.penM = np.nan
        self.scB = self.hiB = self.arB = self.penLB = self.sosB = self.rngB = 0
        self.penB = None
        self.stDone = False
        self.spr = False
        self.d = None
        self.sig = False
        self.sprEv = False
        self.sosEv = False
        self.msg = ""


# ═══════════════════════ ANA MOTOR ═══════════════════════
class Motor:
    def __init__(self, df, ay, sembol, aralik):
        self.ay = ay
        self.sembol = sembol
        self.aralik = aralik
        self.idx = df.index
        self.o = df["open"].to_numpy(float)
        self.h = df["high"].to_numpy(float)
        self.l = df["low"].to_numpy(float)
        self.c = df["close"].to_numpy(float)
        self.v = df["volume"].to_numpy(float) if "volume" in df else np.zeros(len(df))
        self.v = np.nan_to_num(self.v)
        self.n = len(df)
        self.i = 0

        self.fp_ = ay.fm_swing if ay.fm_swing > 0 else ay.swing
        self.atr = atr_rma(self.h, self.l, self.c, 14)
        self.ph = pivot_high(self.h, ay.swing)
        self.pl = pivot_low(self.l, ay.swing)
        self.fph = pivot_high(self.h, self.fp_)
        self.fpl = pivot_low(self.l, self.fp_)
        vs = pd.Series(self.v)
        self.fmVolMa = vs.rolling(ay.fm_hacim_len, min_periods=ay.fm_hacim_len).mean().to_numpy()
        self.wyVolMa = vs.rolling(20, min_periods=20).mean().to_numpy()
        self.wyLoN = pd.Series(self.l).rolling(ay.wy_trend_len, min_periods=ay.wy_trend_len).min().to_numpy()
        self.wyHiN = pd.Series(self.h).rolling(ay.wy_trend_len, min_periods=ay.wy_trend_len).max().to_numpy()
        self.wyEma = pd.Series(self.c).ewm(span=ay.wy_ema, adjust=False).mean().to_numpy()

        # piyasa yapısı durumu
        self.sZP, self.sZB, self.sZT = [], [], []
        self.swLabels = []                    # (bar, fiyat, HH/HL/LH/LL)
        self.kir = []                         # tüm kırılım olayları (BOS/CHoCH, formasyon, Wyckoff)
        self.zz = []                          # zigzag noktaları (bar, fiyat)
        self.bosList = []                     # (bar, seviye_bar, tür, yön, seviye)
        self.lastSLBar = None
        self.lastStruct = "—"
        self.lastBreak = "—"
        self.lastBreakBar = None
        self.swHi = self.swLo = np.nan
        self.swHiBar = self.swLoBar = 0
        self.swHiDone = self.swLoDone = True
        self.trend = 0

        # formasyon durumu
        self.fZP, self.fZB, self.fZT = [], [], []
        self.hist = []                        # son N formasyon kaydı
        self.fmRv = [Rev(), Rev()]            # 0 = düşüş beklenen, 1 = yükseliş beklenen
        self.fmLn = Lin()
        self.fmBlock = 0
        self.fmRevEnd = 0
        self.fmBrkDir = 0
        self.fmBrkBar = None
        self.fmBrkNm = ""
        self.tr = []
        self.stBrk = self.stHit = self.stFail = self.stInv = 0
        self.stHitV = self.stFailV = 0
        self.keys = []
        self.hitMap, self.failMap = {}, {}
        self.fid = 0

        # wyckoff durumu
        self.wyA, self.wyD = Wy(1), Wy(-1)
        self.wyHist = []
        self.wyTr = []
        self.wyN = [0] * 6
        self.wyHit = [0] * 6
        self.wyFail = [0] * 6

        self.olaylar = []                     # (bar, mesaj)

    # ───── genel ─────
    def tarih(self, b):
        try:
            return self.idx[int(b)].strftime("%d.%m.%Y %H:%M")
        except Exception:
            return str(b)

    @property
    def pre(self):
        return f"{self.sembol} {self.aralik}"

    def log(self, msg):
        self.olaylar.append((self.i, msg))

    def hacim_ok(self, i):
        a = self.ay
        m = self.fmVolMa[i]
        return bool(a.fm_hacim and self.v[i] > 0 and not isnan(m) and self.v[i] >= m * a.fm_hacim_kat)

    # ═════════ PİYASA YAPISI ═════════
    def yapi(self, i):
        ay = self.ay
        for t, pv in ((1, self.ph[i]), (-1, self.pl[i])):
            if isnan(pv):
                continue
            pb = i - ay.swing
            n = len(self.sZP)
            ev = 0
            oldB = self.sZB[-1] if n > 0 else None
            if n == 0 or self.sZT[-1] != t:
                self.sZP.append(pv)
                self.sZB.append(pb)
                self.sZT.append(t)
                ev = 1
            elif (t == 1 and pv > self.sZP[-1]) or (t == -1 and pv < self.sZP[-1]):
                self.sZP[-1] = pv
                self.sZB[-1] = pb
                ev = 2
            if len(self.sZP) > 10:
                self.sZP.pop(0)
                self.sZB.pop(0)
                self.sZT.pop(0)
            if ev > 0:
                n = len(self.sZP)
                if ev == 1 or not self.zz:
                    self.zz.append((pb, pv))
                else:
                    self.zz[-1] = (pb, pv)
                if n >= 3:
                    ps = self.sZP[-3]
                    st = ("HH" if pv > ps else "LH") if t == 1 else ("LL" if pv < ps else "HL")
                    self.lastStruct = st
                    if ev == 2 and self.swLabels and self.lastSLBar == oldB:
                        self.swLabels[-1] = (pb, pv, st)
                    else:
                        self.swLabels.append((pb, pv, st))
                        while len(self.swLabels) > ay.yapi_max:
                            self.swLabels.pop(0)
                    self.lastSLBar = pb
                if t == 1:
                    self.swHi, self.swHiBar, self.swHiDone = pv, pb, False
                else:
                    self.swLo, self.swLoBar, self.swLoDone = pv, pb, False

        upS = self.c[i] if ay.bos_kapanis else self.h[i]
        dnS = self.c[i] if ay.bos_kapanis else self.l[i]

        if not self.swHiDone and not isnan(self.swHi) and upS > self.swHi:
            self.swHiDone = True
            ch = self.trend == -1
            self.trend = 1
            k = "CHoCH" if ch else "BOS"
            self.lastBreak = k + " ↑"
            self.lastBreakBar = i
            self.bosList.append((i, self.swHiBar, k, 1, self.swHi))
            self.kir.append(dict(ad=k, dir=1, bar=i, seviye=self.swHi, hedef=None, hacim=False, rec=None))
            self.bosList = self.bosList[-ay.yapi_max:]
            self.log(f"{self.pre} · {k} yükseliş @ {fp(self.c[i])}")

        if not self.swLoDone and not isnan(self.swLo) and dnS < self.swLo:
            self.swLoDone = True
            ch = self.trend == 1
            self.trend = -1
            k = "CHoCH" if ch else "BOS"
            self.lastBreak = k + " ↓"
            self.lastBreakBar = i
            self.bosList.append((i, self.swLoBar, k, -1, self.swLo))
            self.kir.append(dict(ad=k, dir=-1, bar=i, seviye=self.swLo, hedef=None, hacim=False, rec=None))
            self.bosList = self.bosList[-ay.yapi_max:]
            self.log(f"{self.pre} · {k} düşüş @ {fp(self.c[i])}")

    # ═════════ FORMASYONLAR ═════════
    def adlandir(self, kind, t):
        if kind == 4:
            return "Ters Fincan-Kulp" if t == 1 else "Fincan-Kulp"
        if kind == 2:
            return "OBO" if t == 1 else "TOBO"
        if kind == 3:
            return "Üçlü Tepe" if t == 1 else "Üçlü Dip"
        return "İkili Tepe" if t == 1 else "İkili Dip"

    def kaydet(self, rec):
        self.hist.append(rec)
        while len(self.hist) > self.ay.fm_max:
            o = self.hist.pop(0)
            for r in self.fmRv:
                if r.act and r.rec is o:
                    r.act = False
            if self.fmLn.act and self.fmLn.rec is o:
                self.fmLn.act = False

    def yeni_kayit(self, ad, tur, **kw):
        self.fid += 1
        rec = dict(id=self.fid, ad=ad, tur=tur, olustu=self.i, durum="Kırılım bekliyor",
                   kirilim_bar=None, kirilim_yon=0, seviye=None, hedef=None, hacim=False,
                   geri_test=False, sonuc=None, bitis=None, geri_test_bar=None)
        rec.update(kw)
        return rec

    def kapat(self, rec, durum):
        if rec is not None:
            rec["durum"] = durum
            rec["bitis"] = self.i

    def izle(self, yon, b1, p1, b2, p2, hedef, basarisiz, hacim, ad, rec):
        self.tr.append(dict(dir=yon, b1=b1, p1=p1, b2=b2, p2=p2, tgt=hedef, fail=basarisiz,
                            bar=self.i, vol=hacim, nm=ad, rec=rec, rt=self.ay.fm_gt))
        if ad not in self.keys:
            self.keys.append(ad)
        while len(self.tr) > 30:
            self.tr.pop(0)

    def formasyon(self, i):
        ay = self.ay
        fpv = self.fp_
        atr = self.atr[i]
        volOk = self.hacim_ok(i)

        for idx, (t, pv) in enumerate(((1, self.fph[i]), (-1, self.fpl[i]))):
            if isnan(pv):
                continue
            pb = i - fpv
            zn = len(self.fZP)
            fresh = False
            if zn == 0 or self.fZT[-1] != t:
                self.fZP.append(pv)
                self.fZB.append(pb)
                self.fZT.append(t)
                fresh = True
            elif (t == 1 and pv > self.fZP[-1]) or (t == -1 and pv < self.fZP[-1]):
                self.fZP[-1] = pv
                self.fZB[-1] = pb
            if len(self.fZP) > 60:
                self.fZP.pop(0)
                self.fZB.pop(0)
                self.fZT.pop(0)
            if not fresh:
                continue

            zn = len(self.fZP)
            P = lambda k: self.fZP[-1 - k]
            B = lambda k: self.fZB[-1 - k]
            kind, km = 0, 0
            nb1, np1, nb2, np2 = 0, 0.0, 1, 0.0
            cHg = cInv = math.nan
            tol = ay.fm_tol / 100.0
            minH = ay.fm_min_atr * atr

            # ── Üçlü / OBO ──
            if zn >= 5 and (ay.fm_hs or ay.fm_tpl):
                H3, N2, H2, N1, H1 = t * P(0), t * P(1), t * P(2), t * P(3), t * P(4)
                trOk = (not ay.fm_trend) or (zn >= 6 and t * P(5) < min(N1, N2))
                d1, d2 = B(2) - B(4), B(0) - B(2)
                symOk = max(d1, d2) <= ay.fm_sym * max(min(d1, d2), 1)
                nAvg = (N1 + N2) / 2.0
                aboveOk = min(H1, H2, H3) > max(N1, N2)
                hT = (H1 + H2 + H3) / 3.0 - nAvg
                if (ay.fm_tpl and trOk and symOk and aboveOk and hT >= minH
                        and max(H1, H2, H3) - min(H1, H2, H3) <= tol * hT and abs(N1 - N2) <= 0.5 * hT):
                    kind = 3
                    km = 4 if (H1 >= H2 and H1 >= H3) else (2 if H2 >= H3 else 0)
                hH = H2 - nAvg
                if (kind == 0 and ay.fm_hs and trOk and symOk and aboveOk and hH >= minH
                        and H2 - max(H1, H3) >= ay.fm_bas / 100.0 * hH
                        and abs(H1 - H3) <= tol * hH and abs(N1 - N2) <= 0.5 * hH):
                    kind = 2
                    km = 2
                if kind > 0:
                    nb1, np1, nb2, np2 = B(3), P(3), B(1), P(1)

            # ── Fincan-Kulp ──
            if kind == 0 and ay.fm_cup and zn >= 4:
                c = -t
                hnd, rR, bot, rL = P(0), P(1), P(2), P(3)
                dep = c * ((rL + rR) / 2.0 - bot)
                hDep = c * (rR - hnd)
                cupW = B(1) - B(3)
                hW = B(0) - B(1)
                if (dep >= minH and abs(rL - rR) <= tol * dep and hDep > 0 and hDep <= 0.5 * dep
                        and cupW >= 4 * fpv and hW <= 0.5 * cupW):
                    kind, km = 4, 2
                    nb1, np1, nb2, np2 = B(1), rR, B(1) + 1, rR
                    cHg = dep
                    cInv = bot + c * dep * 0.5

            # ── İkili ──
            if kind == 0 and ay.fm_dbl and zn >= 3:
                B2, NN, B1 = t * P(0), t * P(1), t * P(2)
                trOk2 = (not ay.fm_trend) or (zn >= 4 and t * P(3) < NN)
                hD = max(B1, B2) - NN
                if trOk2 and hD >= minH and abs(B1 - B2) <= tol * hD:
                    kind = 1
                    km = 2 if B1 >= B2 else 0
                    nb1, np1, nb2, np2 = B(1), P(1), B(1) + 1, P(1)

            # ── dönüş formasyonu kaydı ──
            if kind > 0:
                ix = 0 if t == 1 else 1
                nm = self.adlandir(kind, t)
                old = self.fmRv[ix]
                if old.act and old.rec is not None:
                    self.kapat(old.rec, "Pasif (yenisi oluştu)")
                nkNow = Y(nb1, np1, nb2, np2, i)
                hg = cHg if kind == 4 else abs(P(km) - Y(nb1, np1, nb2, np2, B(km)))
                inv = cInv if kind == 4 else P(km)
                ks = (3, 2, 1, 0) if kind == 4 else (2, 1, 0) if kind == 1 else (4, 3, 2, 1, 0)
                rec = self.yeni_kayit(nm, "Dönüş", bekle="aşağı" if t == 1 else "yukarı",
                                      boyun_simdi=nkNow, tepe=P(km), inv=inv, kind=kind,
                                      noktalar=[(B(k), P(k)) for k in ks],
                                      boyun=(nb1, np1, nb2, np2), bas_bar=B(km))
                self.fmRv[ix] = Rev(True, nb1, np1, nb2, np2, i, hg, inv, nm, rec)
                self.fmRevEnd = max(self.fmRevEnd, B(0))
                self.kaydet(rec)
                self.log(f"{self.pre} · {nm} oluştu, boyun {fp(nkNow)}")

            # ══ çizgi formasyonları ══
            m = 5 if ay.fm_dokunus == 5 else 4
            poleSkip = False
            if kind == 0 and zn >= m:
                leg0 = abs(P(m - 1) - P(m - 2))
                legMax = 0.0
                for k in range(0, m - 2):
                    legMax = max(legMax, abs(P(k) - P(k + 1)))
                if legMax > 0 and leg0 >= ay.fm_direk * legMax:
                    if m == 5:
                        m = 4
                    else:
                        poleSkip = True

            if kind == 0 and not poleSkip and zn >= m and B(m - 1) >= max(self.fmBlock, self.fmRevEnd):
                kHn = 0 if t == 1 else 1
                kHo = (4 if m == 5 else 2) if t == 1 else 3
                kLn = 1 if t == 1 else 0
                kLo = 3 if t == 1 else (4 if m == 5 else 2)
                uB1, uP1, uB2, uP2 = B(kHo), P(kHo), B(kHn), P(kHn)
                dB1, dP1, dB2, dP2 = B(kLo), P(kLo), B(kLn), P(kLn)
                xs, xe = B(m - 1), B(0)
                W = xe - xs
                Hs = Y(uB1, uP1, uB2, uP2, xs) - Y(dB1, dP1, dB2, dP2, xs)
                He = Y(uB1, uP1, uB2, uP2, xe) - Y(dB1, dP1, dB2, dP2, xe)

                if W > 0 and Hs >= minH and He > 0:
                    slU = (uP2 - uP1) / float(uB2 - uB1)
                    slL = (dP2 - dP1) / float(dB2 - dB1)
                    su = slU * W / Hs
                    sd = slL * W / Hs
                    rat = He / Hs
                    midOk = True
                    if m == 5:
                        midL = Y(uB1, uP1, uB2, uP2, B(2)) if t == 1 else Y(dB1, dP1, dB2, dP2, B(2))
                        midOk = abs(P(2) - midL) <= tol * Hs

                    if midOk:
                        uF, lF = abs(su) < ay.fm_yatay, abs(sd) < ay.fm_yatay
                        uUp, uDn = su >= ay.fm_yatay, su <= -ay.fm_yatay
                        lUp, lDn = sd >= ay.fm_yatay, sd <= -ay.fm_yatay
                        lnm, cat, ldr = "", 0, 0
                        if uF and lF:
                            lnm, cat = "Dikdörtgen", 3
                        elif uF and lUp:
                            lnm, cat, ldr = "Yükselen Üçgen", 1, 1
                        elif uDn and lF:
                            lnm, cat, ldr = "Alçalan Üçgen", 1, -1
                        elif uDn and lUp:
                            lnm, cat = "Simetrik Üçgen", 1
                        elif uUp and lUp:
                            if rat < 0.75:
                                lnm, cat, ldr = "Yükselen Takoz", 2, -1
                            elif rat <= 1.33:
                                lnm, cat = "Yükselen Kanal", 4
                            else:
                                lnm, cat = "Genişleyen Formasyon", 5
                        elif uDn and lDn:
                            if rat < 0.75:
                                lnm, cat, ldr = "Alçalan Takoz", 2, 1
                            elif rat <= 1.33:
                                lnm, cat = "Düşen Kanal", 4
                            else:
                                lnm, cat = "Genişleyen Formasyon", 5
                        else:
                            lnm, cat = "Genişleyen Formasyon", 5

                        # bayrak / flama
                        pole = 0.0
                        if ay.fm_flag and zn >= m + 1 and cat in (1, 3, 4):
                            poleLen = abs(P(m - 1) - P(m))
                            poleBars = B(m - 1) - B(m)
                            poleUp = P(m - 1) > P(m)
                            isTriS = cat == 1 and ldr == 0
                            isRect = cat == 3
                            isChDn = cat == 4 and su < 0
                            isChUp = cat == 4 and su > 0
                            if poleLen >= ay.fm_direk * Hs and poleBars <= W:
                                if poleUp and (isChDn or isRect):
                                    lnm, cat, ldr, pole = "Boğa Bayrağı", 6, 1, poleLen
                                elif poleUp and isTriS:
                                    lnm, cat, ldr, pole = "Boğa Flaması", 6, 1, poleLen
                                elif (not poleUp) and (isChUp or isRect):
                                    lnm, cat, ldr, pole = "Ayı Bayrağı", 6, -1, poleLen
                                elif (not poleUp) and isTriS:
                                    lnm, cat, ldr, pole = "Ayı Flaması", 6, -1, poleLen

                        catOn = {1: ay.fm_tri, 2: ay.fm_wdg, 3: ay.fm_rect, 4: ay.fm_chan,
                                 5: ay.fm_broad, 6: ay.fm_flag}.get(cat, False)
                        if catOn:
                            L = self.fmLn
                            if L.act:
                                if xs < L.ptEnd:
                                    if L.rec in self.hist:
                                        self.hist.remove(L.rec)
                                else:
                                    if L.rec is not None:
                                        self.kapat(L.rec, "Pasif (yenisi oluştu)")
                            arw = " ↑" if ldr == 1 else " ↓" if ldr == -1 else ""
                            apx = (xs + int(round(Hs / (slL - slU)))) if slL > slU else None
                            rec = self.yeni_kayit(lnm + arw, "Çizgi", yon=ldr,
                                                  ust=(uB1, uP1, uB2, uP2), alt=(dB1, dP1, dB2, dP2),
                                                  boy=Hs, direk=pole)
                            self.fmLn = Lin(True, uB1, uP1, uB2, uP2, dB1, dP1, dB2, dP2, i, xe, apx,
                                            ldr, Hs, pole, lnm, rec)
                            self.kaydet(rec)
                            self.log(f"{self.pre} · {lnm}{arw} oluştu")

        # ── dönüş formasyonu boyun kırılımı ──
        for ix in (0, 1):
            r = self.fmRv[ix]
            if not r.act:
                continue
            s = 1 if ix == 0 else -1
            nk = Y(r.b1, r.p1, r.b2, r.p2, i)
            cl = self.c[i]
            if (s == 1 and cl < nk) or (s == -1 and cl > nk):
                r.act = False
                tg = nk - s * r.hgt
                rec = r.rec
                rec.update(durum="Kırıldı", bitis=i, kirilim_bar=i, kirilim_yon=-s, seviye=nk, hedef=tg, hacim=volOk)
                self.fmBrkDir, self.fmBrkBar, self.fmBrkNm = -s, i, r.nm
                self.kir.append(dict(ad=r.nm, dir=-s, bar=i, seviye=nk, hedef=tg, hacim=volOk, rec=rec))
                self.stBrk += 1
                self.izle(-s, r.b1, r.p1, r.b2, r.p2, tg, r.inv, volOk, r.nm, rec)
                self.log(f"{self.pre} · {r.nm} boyun kırıldı ({'aşağı' if s == 1 else 'yukarı'}), hedef {fp(tg)}"
                         + (" · Hacim ✓" if volOk else ""))
            elif (s == 1 and cl > r.inv) or (s == -1 and cl < r.inv) or i - r.st > ay.fm_bekle:
                r.act = False
                self.stInv += 1
                self.kapat(r.rec, "Geçersiz")

        # ── çizgi formasyonu kırılımı ──
        L = self.fmLn
        if L.act:
            uN = Y(L.uB1, L.uP1, L.uB2, L.uP2, i)
            dN = Y(L.dB1, L.dP1, L.dB2, L.dP2, i)
            cl = self.c[i]
            bd = 1 if cl > uN else -1 if cl < dN else 0
            if bd != 0:
                L.act = False
                self.fmBlock = i
                lv = uN if bd == 1 else dN
                tg = lv + bd * (L.pole if L.pole > 0 else L.hgt)
                L.rec.update(durum="Kırıldı", bitis=i, kirilim_bar=i, kirilim_yon=bd, seviye=lv, hedef=tg, hacim=volOk)
                self.fmBrkDir, self.fmBrkBar, self.fmBrkNm = bd, i, L.nm
                self.kir.append(dict(ad=L.nm, dir=bd, bar=i, seviye=lv, hedef=tg, hacim=volOk, rec=L.rec))
                self.stBrk += 1
                if bd == 1:
                    self.izle(1, L.uB1, L.uP1, L.uB2, L.uP2, tg, dN, volOk, L.nm, L.rec)
                else:
                    self.izle(-1, L.dB1, L.dP1, L.dB2, L.dP2, tg, uN, volOk, L.nm, L.rec)
                self.log(f"{self.pre} · {L.nm} {'yukarı' if bd == 1 else 'aşağı'} kırıldı, hedef {fp(tg)}"
                         + (" · Hacim ✓" if volOk else ""))
            elif i - L.st > ay.fm_bekle or (L.apex is not None and i >= L.apex):
                L.act = False
                self.stInv += 1
                self.kapat(L.rec, "Geçersiz (süre doldu / tepe noktası)")

        # ── kırılım sonrası: geri test ve sonuç ──
        for j in range(len(self.tr) - 1, -1, -1):
            q = self.tr[j]
            if i <= q["bar"]:
                continue
            lv = Y(q["b1"], q["p1"], q["b2"], q["p2"], i)
            rTol = ay.fm_gt_tol * atr
            done = False
            cl = self.c[i]
            if q["rt"]:
                if q["dir"] == 1:
                    lost = cl < lv - rTol
                    touched = self.l[i] <= lv + rTol and cl > lv
                else:
                    lost = cl > lv + rTol
                    touched = self.h[i] >= lv - rTol and cl < lv
                if i - q["bar"] > ay.fm_gt_bar or lost:
                    q["rt"] = False
                elif i - q["bar"] >= 2 and touched:
                    q["rt"] = False
                    if q["rec"] is not None:
                        q["rec"]["geri_test"] = True
                        q["rec"]["geri_test_bar"] = i
                    self.log(f"{self.pre} · {q['nm']} geri test ✓ @ {fp(cl)}")
            failed = cl < q["fail"] if q["dir"] == 1 else cl > q["fail"]
            hit = self.h[i] >= q["tgt"] if q["dir"] == 1 else self.l[i] <= q["tgt"]
            if failed:
                self.stFail += 1
                self.stFailV += 1 if q["vol"] else 0
                self.failMap[q["nm"]] = self.failMap.get(q["nm"], 0) + 1
                if q["rec"] is not None:
                    q["rec"]["sonuc"] = "Başarısız ✗"
                done = True
            elif hit:
                self.stHit += 1
                self.stHitV += 1 if q["vol"] else 0
                self.hitMap[q["nm"]] = self.hitMap.get(q["nm"], 0) + 1
                if q["rec"] is not None:
                    q["rec"]["sonuc"] = "Hedefe ulaştı ✓"
                done = True
            elif i - q["bar"] > ay.fm_bekle * 3:
                done = True
            if done:
                self.tr.pop(j)

    # ═════════ WYCKOFF ═════════
    def wy_ad(self, s):
        return "Wyckoff Birikim" if s == 1 else "Wyckoff Dağıtım"

    def wy_etiket(self, w, b, m, metin):
        w.d["olaylar"].append((int(b), w.s * m, metin))

    def wy_gri(self, w, neden):
        w.d["gecersiz"] = neden

    def wy_adim(self, w, o, pLm, pHm):
        ay = self.ay
        i = self.i
        s = w.s
        mL = self.l[i] if s == 1 else -self.h[i]
        mH = self.h[i] if s == 1 else -self.l[i]
        mC = s * self.c[i]
        nExt = (self.l[i] <= self.wyLoN[i]) if s == 1 else (self.h[i] >= self.wyHiN[i])
        pb = i - ay.swing
        nSpr = "Spring" if s == 1 else "UTAD"
        nSOS = "SOS" if s == 1 else "SOW"
        pre = f"{self.pre} · {self.wy_ad(s)}"
        atr = self.atr[i]
        barR = self.h[i] - self.l[i]
        hasVol = self.v[i] > 0 and not isnan(self.wyVolMa[i])
        if hasVol:
            clx = barR >= ay.wy_clx_rng * atr and self.v[i] >= ay.wy_clx_vol * self.wyVolMa[i]
        else:
            clx = barR >= ay.wy_clx_rng * atr and barR >= (ay.wy_clx_rng + 0.5) * atr
        volOk = self.hacim_ok(i)
        oRng = o.st in (2, 3)

        # Faz A
        if (w.st == 0 and clx and nExt and not oRng) or (w.st == 1 and mL < w.scM):
            w.st = 1
            w.scM, w.scB = mL, i
            w.hiM, w.hiB = mH, i
        elif w.st == 1:
            if mH > w.hiM:
                w.hiM, w.hiB = mH, i
            if (not isnan(pHm)) and pb > w.scB and w.hiM - w.scM > ay.wy_max_h * atr:
                w.st = 0
            elif (not isnan(pHm)) and pb > w.scB and w.hiM - w.scM >= ay.wy_min_h * atr and not oRng:
                if o.st == 1:
                    o.st = 0
                w.arM, w.arB, w.rngB = w.hiM, w.hiB, i
                w.st = 2
                w.stDone = False
                w.spr = False
                w.penB = None
                yb, yt = s * w.scM, s * w.arM
                d = dict(s=s, sc_bar=w.scB, ar_bar=w.arB, olustu=i, ust=max(yb, yt), alt=min(yb, yt),
                         sag=i, olaylar=[], gecersiz=None, sinyal=None, hedef=None, sahip=w,
                         sc_fiyat=yb, ar_fiyat=yt)
                w.d = d
                self.wyHist.append(d)
                while len(self.wyHist) > ay.wy_max:
                    self.wyHist.pop(0)
                self.wy_etiket(w, w.scB, w.scM, "SC" if s == 1 else "BC")
                self.wy_etiket(w, w.arB, w.arM, "AR")
                w.msg = pre + ": işlem aralığı oluştu"
            elif i - w.scB > ay.wy_ar_max:
                w.st = 0

        # Faz B / C
        elif w.st in (2, 3):
            h = w.arM - w.scM
            w.d["sag"] = i
            if mC < w.scM - ay.wy_spr_dep * h:
                self.wy_gri(w, "aşağı kırıldı" if s == 1 else "yukarı kırıldı")
                w.st = 0
            elif mC > w.arM:
                ok = w.spr or not ay.wy_spr_sart
                w.sosEv = True
                tg = s * (w.arM + h)
                w.sosB = i
                w.st = 4
                metin = ((("▲ " if s == 1 else "▼ ") if ok else "") + nSOS
                         + ((" · AL" if s == 1 else " · SAT") if ok else "")
                         + (" · Hacim ✓" if volOk else ""))
                self.wy_etiket(w, i, mH, metin)
                w.d["sos_bar"] = i
                if ok:
                    w.sig = True
                    w.d["sinyal"] = "AL" if s == 1 else "SAT"
                    w.d["hedef"] = tg
                    w.msg = (pre + " · " + nSOS + (" → AL @ " if s == 1 else " → SAT @ ") + fp(self.c[i])
                             + ", hedef " + fp(tg) + (" · Hacim ✓" if volOk else ""))
                else:
                    w.d["sinyal"] = "yok (Spring/UTAD görülmedi)"
                    w.msg = pre + " · " + nSOS + " (Spring / UTAD yok, sinyal verilmedi)"
            else:
                if (w.st == 2 and not w.stDone and not isnan(pLm) and pb > w.arB
                        and w.scM - 0.1 * h <= pLm <= w.scM + 0.35 * h):
                    w.stDone = True
                    self.wy_etiket(w, pb, pLm, "ST")
                if w.st == 2 and i - w.rngB >= ay.wy_min_b:
                    if mL < w.scM:
                        if w.penB is None:
                            w.penB, w.penM, w.penLB = i, mL, i
                        elif mL < w.penM:
                            w.penM, w.penLB = mL, i
                    if w.penB is not None:
                        if mC > w.scM and w.penM >= w.scM - ay.wy_spr_dep * h:
                            w.spr = True
                            w.sprEv = True
                            w.st = 3
                            w.penB = None
                            self.wy_etiket(w, w.penLB, w.penM, nSpr)
                            w.msg = pre + " · " + nSpr + " görüldü @ " + fp(self.c[i])
                        elif i - w.penB >= ay.wy_spr_bar:
                            w.penB = None
                if w.st != 0 and i - w.scB > ay.wy_max_bar:
                    self.wy_gri(w, "süre doldu")
                    w.st = 0

        # Faz D / E
        elif w.st == 4:
            h = w.arM - w.scM
            if (not isnan(pLm)) and pb > w.sosB and pLm > w.scM + 0.5 * h:
                self.wy_etiket(w, pb, pLm, "LPS" if s == 1 else "LPSY")
                w.st = 0
            elif mC < w.scM or i - w.sosB > ay.wy_ar_max:
                w.st = 0

    def wy_push(self, w):
        i = self.i
        h = w.arM - w.scM
        al = self.c[i] > self.wyEma[i] if w.s == 1 else self.c[i] < self.wyEma[i]
        grp = 0 if w.spr else (1 if al else 2)
        k = grp * 2 + (0 if w.s == 1 else 1)
        self.wyTr.append(dict(dir=w.s, tgt=w.s * (w.arM + h), fail=w.s * w.scM, k=k, bar=i))
        self.wyN[k] += 1

    def wyckoff(self, i):
        ay = self.ay
        self.wyA.sig = self.wyD.sig = False
        self.wyA.sprEv = self.wyD.sprEv = False
        self.wyA.sosEv = self.wyD.sosEv = False
        self.wyA.msg = self.wyD.msg = ""
        if ay.wy_acik:
            if ay.wy_bir:
                self.wy_adim(self.wyA, self.wyD, self.pl[i], self.ph[i])
            if ay.wy_dag:
                self.wy_adim(self.wyD, self.wyA, -self.ph[i], -self.pl[i])

            # ölçüm: her SOS / SOW
            for j in range(len(self.wyTr) - 1, -1, -1):
                q = self.wyTr[j]
                if i <= q["bar"]:
                    continue
                done = False
                failed = self.c[i] < q["fail"] if q["dir"] == 1 else self.c[i] > q["fail"]
                hit = self.h[i] >= q["tgt"] if q["dir"] == 1 else self.l[i] <= q["tgt"]
                if failed:
                    self.wyFail[q["k"]] += 1
                    done = True
                elif hit:
                    self.wyHit[q["k"]] += 1
                    done = True
                elif i - q["bar"] > ay.wy_max_bar:
                    done = True
                if done:
                    self.wyTr.pop(j)
            if self.wyA.sosEv:
                self.wy_push(self.wyA)
            if self.wyD.sosEv:
                self.wy_push(self.wyD)

            volOk = self.hacim_ok(i)
            for w_, ad_ in ((self.wyA, "Wyckoff AL (SOS)"), (self.wyD, "Wyckoff SAT (SOW)")):
                if w_.sig:
                    self.kir.append(dict(ad=ad_, dir=w_.s, bar=i, seviye=w_.d["ar_fiyat"], hedef=w_.d["hedef"],
                                         hacim=volOk, rec=w_.d))
            if ay.fm_acik and self.wyA.sig:
                w = self.wyA
                h = w.arM - w.scM
                self.stBrk += 1
                self.izle(1, w.arB, w.arM, w.arB + 1, w.arM, w.arM + h, w.scM, volOk, self.wy_ad(1), w.d)
            if ay.fm_acik and self.wyD.sig:
                w = self.wyD
                h = w.arM - w.scM
                self.stBrk += 1
                self.izle(-1, w.arB, -w.arM, w.arB + 1, -w.arM, -(w.arM + h), -w.scM, volOk, self.wy_ad(-1), w.d)

            for w in (self.wyA, self.wyD):
                if w.msg:
                    self.log(w.msg)

    # ═════════ ANA DÖNGÜ ═════════
    def calistir(self):
        for i in range(self.n):
            self.i = i
            self.yapi(i)
            if self.ay.fm_acik:
                self.formasyon(i)
            self.wyckoff(i)
        return self


# ═══════════════════════ RAPOR ═══════════════════════
def wy_faz(w):
    nm = "Birikim" if w.s == 1 else "Dağıtım"
    return {1: f"{nm} · Faz A ({'SC' if w.s == 1 else 'BC'})",
            2: f"{nm} · Faz B",
            3: f"{nm} · Faz C ({'Spring' if w.s == 1 else 'UTAD'})",
            4: f"{nm} · Faz D ({'SOS' if w.s == 1 else 'SOW'})"}.get(w.st, "")


def kutu(satirlar, baslik):
    kw = max(len(k) for k, _ in satirlar)
    vw = max(len(str(v)) for _, v in satirlar)
    gen = max(kw + vw + 5, len(baslik) + 4)
    print("┌" + "─" * gen + "┐")
    print("│ " + baslik.ljust(gen - 1) + "│")
    print("├" + "─" * gen + "┤")
    for k, v in satirlar:
        print("│ " + k.ljust(kw) + "  " + str(v).rjust(gen - kw - 3) + " │")
    print("└" + "─" * gen + "┘")


def form_satiri(m, f, i_son):
    t = m.tarih
    s = f"• {f['ad']}  [{f['tur']}]  oluştu: {t(f['olustu'])}"
    if f["durum"] == "Kırılım bekliyor":
        if f["tur"] == "Dönüş":
            s += f"\n    Durum: BOYUN KIRILIMI BEKLİYOR ({f['bekle']} yönde) · boyun şu an ≈ {fp(f['boyun_simdi'])}"
            s += f" · baş/tepe {fp(f['tepe'])}"
        else:
            s += "\n    Durum: KIRILIM BEKLİYOR (üst / alt çizgi)"
    elif f["durum"] == "Kırıldı":
        yon = "▲ yukarı" if f["kirilim_yon"] == 1 else "▼ aşağı"
        s += (f"\n    Durum: KIRILDI {yon} · {t(f['kirilim_bar'])} ({i_son - f['kirilim_bar']} bar önce)"
              f" · seviye {fp(f['seviye'])} · hedef {fp(f['hedef'])}")
        s += "\n    " + ("Hacim ✓ · " if f["hacim"] else "") + ("Geri test ✓ · " if f["geri_test"] else "")
        s += "Sonuç: " + (f["sonuc"] or "açık (hedef / ters kapanış bekleniyor)")
    else:
        s += f"\n    Durum: {f['durum']}"
    return s


def rapor(m):
    ay = m.ay
    i_son = m.n - 1
    t = m.tarih
    print("\n" + "═" * 78)
    print(f"  YAPI AVCISI · {m.sembol} · {m.aralik}")
    print(f"  Son kapanmış mum: {t(i_son)}   Kapanış: {fp(m.c[i_son])}   (işlenen bar: {m.n})")
    print("═" * 78)

    # ── yeni olaylar ──
    yeni = [(b, s) for b, s in m.olaylar if b >= i_son - SON_BAR_PENCERESI + 1]
    print(f"\n▌ SON {SON_BAR_PENCERESI} BARDAKİ OLAYLAR / ALARM MESAJLARI")
    if not yeni:
        print("  (bu pencerede yeni olay yok)")
    for b, s in yeni:
        etiket = "🔔 YENİ" if b == i_son else f"{i_son - b} bar önce"
        print(f"  [{t(b)} · {etiket}] {s}")

    # ── bilgi tablosu ──
    rDn, rUp, L = m.fmRv[0], m.fmRv[1], m.fmLn
    trend = "Yükseliş ▲" if m.trend == 1 else "Düşüş ▼" if m.trend == -1 else "—"
    upTxt = f"{rUp.nm} · boyun {fp(Y(rUp.b1, rUp.p1, rUp.b2, rUp.p2, i_son))}" if rUp.act else "—"
    dnTxt = f"{rDn.nm} · boyun {fp(Y(rDn.b1, rDn.p1, rDn.b2, rDn.p2, i_son))}" if rDn.act else "—"
    linTxt = (L.nm + (" ↑" if L.dir == 1 else " ↓" if L.dir == -1 else "")) if L.act else "—"
    brk = "—" if m.fmBrkBar is None else \
        f"{'▲' if m.fmBrkDir == 1 else '▼'} {m.fmBrkNm} · {i_son - m.fmBrkBar} bar önce"
    stRes = m.stHit + m.stFail
    stTxt = "—" if m.stBrk == 0 else \
        f"{m.stBrk} kırılım · {m.stHit} ✓ · {m.stFail} ✗ · {yuzde(m.stHit, stRes)}"
    wyPA = wy_faz(m.wyA) if ay.wy_bir else ""
    wyPD = wy_faz(m.wyD) if ay.wy_dag else ""
    wyTxt = (wyPA + (" · " if wyPA and wyPD else "") + wyPD) or "—"

    def grp(g):
        h = m.wyHit[g * 2] + m.wyHit[g * 2 + 1]
        f = m.wyFail[g * 2] + m.wyFail[g * 2 + 1]
        return f"{h}✓ {f}✗"

    satirlar = [
        ("Trend", trend),
        ("Son yapı", m.lastStruct),
        ("Son kırılım", m.lastBreak + ("" if m.lastBreakBar is None else f" ({t(m.lastBreakBar)})")),
        ("Yükseliş formasyonu", upTxt),
        ("Düşüş formasyonu", dnTxt),
        ("Çizgi formasyonu", linTxt),
        ("Son formasyon kırılımı", brk),
    ]
    if ay.fm_stat:
        satirlar.append(("Formasyon başarısı", stTxt))
    if ay.wy_acik:
        satirlar.append(("Wyckoff", wyTxt))
        if ay.wy_olc:
            satirlar.append(("Wyckoff ölçüm", f"S {grp(0)} · T {grp(1)} · K {grp(2)}"))
    print()
    kutu(satirlar, f"YAPI AVCISI · {m.sembol}")

    # ── başarı dökümü ──
    if ay.fm_stat and m.stBrk:
        print("\n▌ FORMASYON BAŞARI DÖKÜMÜ  (✓ hedefe ulaştı · ✗ formasyon ötesine geri kapandı)")
        for k in m.keys:
            h, f = m.hitMap.get(k, 0), m.failMap.get(k, 0)
            if h + f > 0:
                print(f"  {k:<24} {h} ✓ / {f} ✗ · {yuzde(h, h + f)}")
        hN, fN = m.stHit - m.stHitV, m.stFail - m.stFailV
        print(f"  Hacim ✓ kırılımlar     {m.stHitV} ✓ / {m.stFailV} ✗ · {yuzde(m.stHitV, m.stHitV + m.stFailV)}")
        print(f"  Hacimsiz kırılımlar    {hN} ✓ / {fN} ✗ · {yuzde(hN, hN + fN)}")
        print(f"  Kırılmadan geçersiz olan: {m.stInv}")

    # ── formasyonlar ──
    print(f"\n▌ SON {ay.fm_max} FORMASYON (yeniden eskiye)")
    if not m.hist:
        print("  (formasyon bulunamadı)")
    for f in reversed(m.hist):
        print(form_satiri(m, f, i_son))

    # ── Wyckoff ──
    if ay.wy_acik:
        print(f"\n▌ WYCKOFF ŞEMALARI (son {ay.wy_max}, yeniden eskiye)")
        if not m.wyHist:
            print("  (şema bulunamadı)")
        for d in reversed(m.wyHist):
            ad = "Birikim" if d["s"] == 1 else "Dağıtım"
            w = d["sahip"]
            if d["gecersiz"]:
                durum = f"✗ GEÇERSİZ ({d['gecersiz']})"
            elif w.d is d and w.st > 0:
                durum = "AKTİF · " + wy_faz(w)
            else:
                durum = "Tamamlandı"
            print(f"• Wyckoff {ad} · {t(d['olustu'])} · aralık {fp(d['alt'])} – {fp(d['ust'])} · {durum}")
            for b, fy, tx in sorted(d["olaylar"]):
                print(f"    {t(b)}  {tx:<22} @ {fp(fy)}")
            if d["sinyal"]:
                ek = f" · hedef {fp(d['hedef'])}" if d["hedef"] is not None else ""
                print(f"    Sinyal: {d['sinyal']}{ek}" + (f" · Sonuç: {d.get('sonuc')}" if d.get("sonuc") else ""))

    # ── yapı etiketleri ──
    print("\n▌ SON PİYASA YAPISI ETİKETLERİ (yeniden eskiye, son 8)")
    for b, fy, tx in reversed(m.swLabels[-8:]):
        print(f"  {t(b)}  {tx}  @ {fp(fy)}")
    print("\n▌ SON BOS / CHoCH (yeniden eskiye, son 8)")
    for b, sb, k, yon, lv in reversed(m.bosList[-8:]):
        print(f"  {t(b)}  {k} {'↑' if yon == 1 else '↓'}  seviye {fp(lv)}")
    print()


# ═══════════════════════ VERİ ═══════════════════════
def veri_cek(tv, Interval, sembol, borsa, aralik, bar):
    if aralik not in ARALIKLAR:
        raise ValueError(f"Geçersiz aralık: {aralik}. Seçenekler: {', '.join(ARALIKLAR)}")
    iv = getattr(Interval, ARALIKLAR[aralik])
    df = tv.get_hist(symbol=sembol, exchange=borsa, interval=iv, n_bars=bar)
    if df is None or len(df) == 0:
        raise RuntimeError(f"{borsa}:{sembol} için veri alınamadı (sembol / borsa adını kontrol edin).")
    df.columns = [c.lower() for c in df.columns]
    for c in ("open", "high", "low", "close"):
        if c not in df:
            raise RuntimeError(f"Eksik sütun: {c}")
    if "volume" not in df:
        df["volume"] = 0.0
    df = df[~df.index.duplicated(keep="last")].sort_index()

    # ── Saat dilimi: indeks HER ZAMAN saat dilimsiz (naive) yerel saate çevrilir ──
    tz = globals().get("SAAT_DILIMI", "Europe/Istanbul") or "UTC"
    idx = pd.DatetimeIndex(df.index)
    if idx.tz is not None:
        idx = idx.tz_convert(tz).tz_localize(None)                      # tz'li (örn. UTC) → yerel saat, tz bilgisini at
    elif aralik not in ("1D", "1W", "1M"):
        idx = idx.tz_localize("UTC").tz_convert(tz).tz_localize(None)   # tz'siz gün içi → UTC kabul et
    df.index = idx

    # ── Haftalık ve aylık barların tarihini gerçek kapanış gününe kaydır ──
    if aralik == "1W":
        df.index = df.index + pd.to_timedelta((4 - df.index.weekday) % 7, unit="D")   # → Cuma
    elif aralik == "1M":
        from pandas.tseries.offsets import MonthEnd
        df.index = df.index + MonthEnd(0)                                              # → ayın son günü

    if not ACIK_MUMU_DAHIL and len(df) > 1:
        df = df.iloc[:-1]       # oluşmakta olan mum
    df.index.name = "yerel_saat"    # çevrildi işareti (eski sürümdeki sarmalayıcı ikinci kez çevirmesin)
    return df


def analiz(df, sembol, aralik, ay=None):
    ay = ay or Ayar()
    return Motor(df, ay, sembol, aralik).calistir()


## 3 · Grafik ve tablo fonksiyonları

In [3]:
# ═══════════════════════ PLOTLY GRAFİĞİ ═══════════════════════
import plotly.graph_objects as go

R_YUK, R_DUS = "#089981", "#F23645"
R_TOP, R_DIP, R_CIZ = "#F57C00", "#1E5FE0", "#9C27B0"
R_ACC, R_DAG, R_GRI = "#00897B", "#E65100", "#9E9E9E"


def _rgba(h, a):
    h = h.lstrip("#")
    r, g, b = (int(h[i:i + 2], 16) for i in (0, 2, 4))
    return f"rgba({r},{g},{b},{a})"


def grafik(m, goster_bar=400, hafta_sonu_gizle=False):
    """Etkileşimli (yakınlaştır / uzaklaştır / kaydır) Plotly grafiği döndürür."""
    n, idx = m.n, m.idx
    bas = max(0, n - goster_bar)
    step = (idx[-1] - idx[-2]) if n > 1 else pd.Timedelta(days=1)

    def X(b):
        b = int(b)
        return idx[b] if b < n else idx[-1] + step * (b - (n - 1))

    fig = go.Figure()
    fig.add_trace(go.Candlestick(
        x=idx[bas:], open=m.o[bas:], high=m.h[bas:], low=m.l[bas:], close=m.c[bas:], name=m.sembol,
        increasing_line_color=R_YUK, decreasing_line_color=R_DUS, showlegend=False))

    # ── zigzag ──
    zz = [(b, p) for b, p in m.zz if b >= bas]
    if m.ay.fm_acik or True:
        fig.add_trace(go.Scatter(x=[X(b) for b, _ in zz], y=[p for _, p in zz], mode="lines",
                                 line=dict(color="rgba(90,90,90,.55)", width=1), name="Zigzag",
                                 hoverinfo="skip"))

    # ── HH / HL / LH / LL ──
    for ad, grup, renk in (("HH · HL", ("HH", "HL"), R_YUK), ("LH · LL", ("LH", "LL"), R_DUS)):
        pts = [(b, p, t) for b, p, t in m.swLabels if t in grup and b >= bas]
        if pts:
            fig.add_trace(go.Scatter(
                x=[X(b) for b, _, _ in pts], y=[p for _, p, _ in pts], mode="text",
                text=[t for _, _, t in pts], textfont=dict(color=renk, size=10),
                textposition=["top center" if t in ("HH", "LH") else "bottom center" for _, _, t in pts],
                name=ad, hoverinfo="skip"))

    # ── BOS / CHoCH ──
    for ad, yon, renk in (("BOS/CHoCH ↑", 1, R_YUK), ("BOS/CHoCH ↓", -1, R_DUS)):
        xs, ys, tx = [], [], []
        for b, sb, k, y_, lv in m.bosList:
            if y_ != yon or b < bas:
                continue
            mid = (max(sb, bas) + b) // 2
            xs += [X(max(sb, bas)), X(mid), X(b), None]
            ys += [lv, lv, lv, None]
            tx += ["", k, "", ""]
        if xs:
            fig.add_trace(go.Scatter(x=xs, y=ys, text=tx, mode="lines+text",
                                     line=dict(color=renk, width=1, dash="dash"),
                                     textposition="top center" if yon == 1 else "bottom center",
                                     textfont=dict(color=renk, size=10), name=ad, hoverinfo="skip"))

    # ── formasyonlar ──
    def kirilim(f, key):
        kb = f["kirilim_bar"]
        if kb is None:
            return
        yon, lv, tg = f["kirilim_yon"], f["seviye"], f["hedef"]
        col = R_YUK if yon == 1 else R_DUS
        txt = ("▲ " if yon == 1 else "▼ ") + f["ad"] + " kırıldı" + (" · Hacim ✓" if f["hacim"] else "")
        ek = "Sonuç: " + (f["sonuc"] or "açık")
        fig.add_trace(go.Scatter(
            x=[X(kb)], y=[lv], mode="markers+text", text=[txt], textfont=dict(color=col, size=10),
            textposition="top center" if yon == 1 else "bottom center",
            marker=dict(symbol="triangle-up" if yon == 1 else "triangle-down", size=11, color=col),
            hovertext=[f"{txt}<br>Seviye {fp(lv)} · Hedef {fp(tg)}<br>{ek}"], hoverinfo="text",
            legendgroup=key, showlegend=False))
        if tg is not None and fm_hedef_ciz:
            fig.add_trace(go.Scatter(
                x=[X(kb), X(kb), X(kb + 20)], y=[lv, tg, tg], mode="lines+text",
                text=["", "", "Hedef " + fp(tg)], textposition="middle right",
                textfont=dict(color=col, size=10),
                line=dict(color=col, width=1.5, dash="dot"), legendgroup=key, showlegend=False,
                hoverinfo="skip"))
        gt = f.get("geri_test_bar")
        if gt is not None:
            fig.add_trace(go.Scatter(
                x=[X(gt)], y=[m.l[gt] if yon == 1 else m.h[gt]], mode="markers+text", text=["Geri test ✓"],
                textposition="bottom center" if yon == 1 else "top center",
                textfont=dict(color=col, size=10), marker=dict(symbol="diamond", size=8, color=col),
                legendgroup=key, showlegend=False, hoverinfo="skip"))

    fm_hedef_ciz = m.ay.fm_hedef
    for f in m.hist:
        son = f["bitis"] if f["bitis"] is not None else n - 1
        if son < bas:
            continue
        gri = f["durum"].startswith(("Geçersiz", "Pasif"))
        key = f"{f['ad']} · {m.tarih(f['olustu'])}"
        if f["tur"] == "Dönüş":
            col = R_GRI if gri else (R_TOP if f["bekle"] == "aşağı" else R_DIP)
            pts = f["noktalar"]
            fig.add_trace(go.Scatter(
                x=[X(b) for b, _ in pts], y=[p for _, p in pts], mode="lines",
                line=dict(color=col, width=2, shape="spline" if f["kind"] == 4 else "linear", smoothing=1.2),
                fill="toself", fillcolor=_rgba(col, .10), name=key, legendgroup=key, showlegend=True,
                hovertext=f"{f['ad']} · {f['durum']}", hoverinfo="text"))
            b1, p1, b2, p2 = f["boyun"]
            fig.add_trace(go.Scatter(
                x=[X(b1), X(son)], y=[p1, Y(b1, p1, b2, p2, son)], mode="lines",
                line=dict(color=col, width=2, dash="dot" if gri else "solid"),
                legendgroup=key, showlegend=False, hoverinfo="skip"))
            ust = f["bekle"] == "aşağı"
            fig.add_trace(go.Scatter(
                x=[X(f["bas_bar"])], y=[f["tepe"]], mode="text", text=[f["ad"]],
                textposition="top center" if ust else "bottom center",
                textfont=dict(color=col, size=12), legendgroup=key, showlegend=False, hoverinfo="skip"))
        else:
            col = R_GRI if gri else R_CIZ
            uB1, uP1, uB2, uP2 = f["ust"]
            dB1, dP1, dB2, dP2 = f["alt"]
            fig.add_trace(go.Scatter(
                x=[X(dB1), X(son)], y=[dP1, Y(dB1, dP1, dB2, dP2, son)], mode="lines",
                line=dict(color=col, width=2, dash="dot" if gri else "solid"),
                legendgroup=key, showlegend=False, hoverinfo="skip"))
            fig.add_trace(go.Scatter(
                x=[X(uB1), X(son)], y=[uP1, Y(uB1, uP1, uB2, uP2, son)], mode="lines",
                line=dict(color=col, width=2, dash="dot" if gri else "solid"),
                fill="tonexty", fillcolor=_rgba(col, .10), name=key, legendgroup=key, showlegend=True,
                hovertext=f"{f['ad']} · {f['durum']}", hoverinfo="text"))
            lx = min(uB1, dB1)
            fig.add_trace(go.Scatter(
                x=[X(lx)], y=[Y(uB1, uP1, uB2, uP2, lx)], mode="text", text=[f["ad"]],
                textposition="top center", textfont=dict(color=col, size=12),
                legendgroup=key, showlegend=False, hoverinfo="skip"))
        kirilim(f, key)

    # ── Wyckoff ──
    for d in m.wyHist:
        if d["sag"] < bas:
            continue
        gri = bool(d["gecersiz"])
        col = R_GRI if gri else (R_ACC if d["s"] == 1 else R_DAG)
        ad = "Wyckoff " + ("Birikim" if d["s"] == 1 else "Dağıtım")
        key = f"{ad} · {m.tarih(d['olustu'])}"
        fig.add_shape(type="rect", x0=X(d["sc_bar"]), x1=X(d["sag"]), y0=d["alt"], y1=d["ust"],
                      line=dict(color=_rgba(col, .7), width=1.5, dash="dot" if gri else "solid"),
                      fillcolor=_rgba(col, .08), layer="below")
        orta = (d["ust"] + d["alt"]) / 2
        ol = [(b, fy, tx) for b, fy, tx in sorted(d["olaylar"]) if b >= bas]
        if ol:
            fig.add_trace(go.Scatter(
                x=[X(b) for b, _, _ in ol], y=[fy for _, fy, _ in ol], mode="markers+text",
                text=[tx for _, _, tx in ol], textfont=dict(color=col, size=10),
                textposition=["bottom center" if fy < orta else "top center" for _, fy, _ in ol],
                marker=dict(size=5, color=col), name=key, legendgroup=key, showlegend=True,
                hovertext=[f"{tx} @ {fp(fy)}" for _, fy, tx in ol], hoverinfo="text"))
        if gri:
            fig.add_annotation(x=X(d["sag"]), y=d["ust"], text="✗ " + d["gecersiz"], showarrow=False,
                               font=dict(color=R_GRI, size=10), xanchor="left", yanchor="bottom")
        if d.get("sinyal") in ("AL", "SAT") and d.get("hedef") is not None and "sos_bar" in d:
            sb, lv, tg = d["sos_bar"], d["ar_fiyat"], d["hedef"]
            fig.add_trace(go.Scatter(
                x=[X(sb), X(sb), X(sb + 20)], y=[lv, tg, tg], mode="lines+text", text=["", "", "Hedef " + fp(tg)],
                textposition="middle right", textfont=dict(color=col, size=10),
                line=dict(color=col, width=1.5, dash="dot"), legendgroup=key, showlegend=False,
                hoverinfo="skip"))

    # ── özet kutusu ──
    trend = "Yükseliş ▲" if m.trend == 1 else "Düşüş ▼" if m.trend == -1 else "—"
    akt = []
    for r in m.fmRv:
        if r.act:
            akt.append(r.nm)
    if m.fmLn.act:
        akt.append(m.fmLn.nm)
    wy = " · ".join(x for x in (wy_faz(m.wyA), wy_faz(m.wyD)) if x) or "—"
    ozet = (f"<b>Trend:</b> {trend}<br><b>Son yapı:</b> {m.lastStruct} · {m.lastBreak}<br>"
            f"<b>Aktif formasyon:</b> {', '.join(akt) if akt else '—'}<br><b>Wyckoff:</b> {wy}")
    fig.add_annotation(xref="paper", yref="paper", x=0.005, y=0.995, text=ozet, showarrow=False,
                       align="left", xanchor="left", yanchor="top", bgcolor="rgba(255,255,255,.88)",
                       bordercolor="#C8C8C0", borderwidth=1, font=dict(size=11, color="#222222"))

    fig.update_layout(
        template="plotly_white", paper_bgcolor="#F6F5F0", plot_bgcolor="#FAF9F5", font=dict(color="#222222"),
        height=820, dragmode="pan", hovermode="closest",
        title=f"Yapı Avcısı · {m.sembol} · {m.aralik} · son mum {m.tarih(n - 1)} · kapanış {fp(m.c[-1])}",
        xaxis_rangeslider_visible=False, margin=dict(l=40, r=60, t=60, b=40),
        legend=dict(orientation="h", y=-0.06, font=dict(size=10)))
    fig.update_xaxes(range=[X(bas), X(n + 25)], gridcolor="#E4E2DA")
    fig.update_yaxes(gridcolor="#E4E2DA")
    if hafta_sonu_gizle:
        fig.update_xaxes(rangebreaks=[dict(bounds=["sat", "mon"])])
    return fig


def tablo_df(m):
    """Formasyon ve Wyckoff kayıtlarını tablo (DataFrame) olarak verir."""
    t = m.tarih
    rows = []
    for f in reversed(m.hist):
        kb = f["kirilim_bar"]
        rows.append({
            "Tür": f["tur"], "Formasyon": f["ad"], "Oluşma": t(f["olustu"]), "Durum": f["durum"],
            "Kırılım zamanı": t(kb) if kb is not None else "",
            "Kırılım yönü": {1: "Yukarı", -1: "Aşağı"}.get(f["kirilim_yon"], ""),
            "Seviye": f["seviye"], "Hedef": f["hedef"],
            "Hacim ✓": "✓" if f["hacim"] else "", "Geri test ✓": "✓" if f["geri_test"] else "",
            "Sonuç": f["sonuc"] or ("açık" if kb is not None else "")})
    for d in reversed(m.wyHist):
        w = d["sahip"]
        durum = ("✗ " + d["gecersiz"]) if d["gecersiz"] else (wy_faz(w) if w.d is d and w.st > 0 else "Tamamlandı")
        rows.append({
            "Tür": "Wyckoff", "Formasyon": "Wyckoff " + ("Birikim" if d["s"] == 1 else "Dağıtım"),
            "Oluşma": t(d["olustu"]), "Durum": durum, "Kırılım zamanı": t(d["sos_bar"]) if "sos_bar" in d else "",
            "Kırılım yönü": d["sinyal"] or "", "Seviye": d["alt"] if d["s"] == 1 else d["ust"],
            "Hedef": d["hedef"], "Hacim ✓": "", "Geri test ✓": "", "Sonuç": d.get("sonuc") or ""})
    return pd.DataFrame(rows)


## 4 · Ayarlar

In [4]:
# ═══════════════ AYARLAR – burayı düzenleyip ilgili hücreleri çalıştırın ═══════════════
#ARALIK = "1D"                # tarama periyodu: 1m 3m 5m 15m 30m 45m 1h 2h 3h 4h 1D 1W 1M

from datetime import datetime
from zoneinfo import ZoneInfo

gun = datetime.now(ZoneInfo("Europe/Istanbul")).isoweekday()   # 1=Pzt … 7=Paz
if gun < 7:
    ARALIK = "1D"        # Pazartesi-Cuma → günlük
    dosya='Gunluk'
elif gun == 7:
    ARALIK = "1W"        # Cumartesi → haftalık
    dosya='Haftalık'
else:
    raise SystemExit("Pazar günü çalışmaz")

BAR_SAYISI = 300             # hisse başına çekilecek bar (en fazla 5000; formasyon istatistiği için yeterli)
SAAT_DILIMI = "Europe/Istanbul"   # gün içi (1h-4h) bar saatleri UTC+3'e çevrilir; çevirmemek için "" yazın
ACIK_MUMU_DAHIL = True      # False: oluşmakta olan son mum atılır (repaint yok)
MIN_BAR = 150                # analiz için gereken en az bar sayısı
ALT_PERIYOTA_IN = True       # bar yetersizse hisse için bir alt periyoda in (1M→1W→1D→4h→3h→2h→1h); en düşük 1h

LISTE_BAR = 0                # sondaki "▲ Yukarı / ▼ Aşağı kırılanlar" sütunları: bar önce ≤ bu değer olanlar (0 = yalnız son kapanmış mum)
TARAMA_YENI_BAR = 3          # Excel'deki "Yeni Olaylar" sayfası: son kaç barda olan olaylar listelensin
YAKINLIK_YUZDE = 2.0         # bekleyen formasyonun boyun / çizgisine uzaklığı bu yüzdenin altındaysa "kırılıma yakın"
BEKLEME = 0.3                # hisseler arası bekleme (sn) – TradingView'i yormamak için
MAX_HISSE = None             # hızlı deneme için örn. 20 yazın; None = tüm liste

SON_BAR_PENCERESI = 10       # tek hisse raporunda "son olaylar" penceresi
GRAFIK_BAR = 300             # grafikte gösterilecek son bar
HAFTA_SONU_GIZLE = True      # BIST için True (hafta sonu boşluklarını gizler)
TV_KULLANICI = ""            # isteğe bağlı TradingView girişi
TV_SIFRE = ""

# Pine ayarlarını değiştirmek için örnek:  AYAR = Ayar(swing=7, fm_max=15, wy_spr_dep=0.25)
AYAR = Ayar()


## 5 · Hisse listesi (TradingView Screener)

In [5]:
# BIST'in TÜM hisselerini TradingView Screener'dan çek → Hisseler listesi
from tradingview_screener import Query, Column

piyasa = 'turkey'
raw = (Query().set_markets(piyasa)
       .select('name', 'type')
       .where(Column('type').isin(['stock']))
       .limit(20000)              # varsayılan 50'yi aş — tüm evren gelsin
       .order_by('name')
       .get_scanner_data())

# get_scanner_data() → (satir_sayisi, veri_listesi)
if isinstance(raw, tuple) and len(raw) == 2:
    rows_data = raw[1]
else:
    rows_data = raw  # fallback

df_raw = pd.DataFrame(rows_data, columns=['ticker', 'name', 'type'])

Hisseler = df_raw['ticker'].astype(str).str.replace('BIST:', '', regex=False).tolist()
print(f'{len(Hisseler)} hisse yüklendi.  Örnek: {Hisseler[:10]} …')


626 hisse yüklendi.  Örnek: ['A1CAP', 'A1YEN', 'AAGYO', 'ACSEL', 'ADEL', 'ADESE', 'ADGYO', 'AEFES', 'AFYON', 'AGESA'] …


## 6 · Tarama ve Excel fonksiyonları

In [6]:
import time
from tvDatafeed import TvDatafeed, Interval
from tqdm.auto import tqdm
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.comments import Comment
from openpyxl.utils import get_column_letter
try:
    from IPython.display import display
except ImportError:
    display = print

tv = TvDatafeed(TV_KULLANICI, TV_SIFRE) if TV_KULLANICI and TV_SIFRE else TvDatafeed()

# Her yapı türü için ayrı sütun çifti açılır: "<Yapı>" = ▲/▼ + tarih, "<Yapı> (bar önce)"
KATEGORILER = ["BOS", "CHoCH", "OBO-TOBO", "Üçlü Tepe-Dip", "İkili Tepe-Dip", "Fincan-Kulp",
               "Üçgen", "Takoz", "Dikdörtgen", "Kanal", "Bayrak-Flama", "Genişleyen", "Wyckoff"]
TEMEL = ["Hisse", "Periyot", "Son mum", "Açılış", "Yüksek", "Düşük", "Kapanış", "Hacim", "Değişim %",
         "Trend", "Son yapı", "Aktif formasyon", "Kırılıma yakınlık %", "Wyckoff faz"]
HIYERARSI = ["1h", "2h", "3h", "4h", "1D", "1W", "1M"]     # en düşük 1h
_SON = {"mum": None}                                        # taranan en yeni mum (Timestamp)
SON_UP, SON_DN = "▲ Yukarı kırılanlar", "▼ Aşağı kırılanlar"
SUTUNLAR = TEMEL + [x for k in KATEGORILER for x in (k, k + " (bar önce)")] + [SON_UP, SON_DN]


def kategori(ad):
    if ad in ("BOS", "CHoCH"):
        return ad
    if ad.startswith("Wyckoff"):
        return "Wyckoff"
    if ad in ("OBO", "TOBO"):
        return "OBO-TOBO"
    for anahtar, k in (("Üçlü", "Üçlü Tepe-Dip"), ("İkili", "İkili Tepe-Dip"), ("Fincan", "Fincan-Kulp"),
                       ("Üçgen", "Üçgen"), ("Takoz", "Takoz"), ("Dikdörtgen", "Dikdörtgen"),
                       ("Kanal", "Kanal"), ("Bayrak", "Bayrak-Flama"), ("Flama", "Bayrak-Flama"),
                       ("Genişleyen", "Genişleyen")):
        if anahtar in ad:
            return k
    return ad


def periyot_zinciri(ar):
    """ARALIK'tan başlayıp bir altına, en fazla 1h'ye kadar inen periyot listesi."""
    if ALT_PERIYOTA_IN and ar in HIYERARSI:
        return HIYERARSI[:HIYERARSI.index(ar) + 1][::-1]
    return [ar]


def hisse_cek(sembol, borsa="BIST", aralik=None, deneme_sayisi=3):
    aralik = aralik or ARALIK
    son = None
    for deneme in range(deneme_sayisi):
        try:
            return veri_cek(tv, Interval, sembol, borsa, aralik, BAR_SAYISI)
        except Exception as e:
            son = e
            time.sleep(0.5 * (deneme + 1))
    raise son


def hisse_cek_oto(sembol, borsa="BIST"):
    """ARALIK'ta MIN_BAR yoksa bir alt periyoda iner; yeterli bar bulunca (df, kullanılan_periyot) döner."""
    log = []
    for k, ar in enumerate(periyot_zinciri(ARALIK)):
        try:
            df = hisse_cek(sembol, borsa, ar, 3 if k == 0 else 2)
        except Exception as e:
            log.append(f"{ar}: veri yok")
            continue
        if len(df) >= MIN_BAR:
            return df, ar
        log.append(f"{ar}={len(df)}")
    raise RuntimeError(f"yetersiz veri (min {MIN_BAR}) · " + ", ".join(log))


def _son_mum_txt():
    return _SON["mum"].strftime("%d-%m-%Y %H:%M") if _SON["mum"] is not None else "-"


def ozet_satir(m, hisse, K, L0):
    """Bir hissenin çalıştığımız andaki en son durumunu tek satırda özetler."""
    n = m.n
    i_son = n - 1
    c = m.c[i_son]
    fmt = "%d-%m-%y" if m.aralik in ("1D", "1W", "1M") else "%d-%m-%y %H:%M"
    tf = lambda b: m.idx[int(b)].strftime(fmt)
    ok = lambda e: "▲ " if e["dir"] == 1 else "▼ "

    # bekleyen formasyonlar ve kırılıma yakınlık
    akt, yak = [], []
    for ix, r in enumerate(m.fmRv):
        if r.act:
            akt.append(f"{r.nm} {'↓' if ix == 0 else '↑'}")
            yak.append(abs(c - Y(r.b1, r.p1, r.b2, r.p2, i_son)) / c * 100)
    L = m.fmLn
    if L.act:
        akt.append(L.nm + (" ↑" if L.dir == 1 else " ↓" if L.dir == -1 else ""))
        uN = Y(L.uB1, L.uP1, L.uB2, L.uP2, i_son)
        dN = Y(L.dB1, L.dP1, L.dB2, L.dP2, i_son)
        yak.append(min(abs(uN - c), abs(c - dN)) / c * 100)

    c0 = m.c[i_son - 1] if n > 1 else None
    row = {
        "Hisse": hisse, "Periyot": m.aralik, "Son mum": tf(i_son),
        "Açılış": m.o[i_son], "Yüksek": m.h[i_son], "Düşük": m.l[i_son], "Kapanış": c, "Hacim": m.v[i_son],
        "Değişim %": (c / c0 - 1) if c0 else None,
        "Trend": "Yükseliş ▲" if m.trend == 1 else "Düşüş ▼" if m.trend == -1 else "—",
        "Son yapı": m.lastStruct,
        "Aktif formasyon": ", ".join(akt),
        "Kırılıma yakınlık %": round(min(yak), 2) if yak else None,
        "Wyckoff faz": " · ".join(x for x in (wy_faz(m.wyA), wy_faz(m.wyD)) if x),
    }

    # her yapı türünün EN SON kırılımı
    son, notlar = {}, {}
    for e in m.kir:
        k = kategori(e["ad"])
        if k not in son or e["bar"] >= son[k]["bar"]:
            son[k] = e
    for k in KATEGORILER:
        e = son.get(k)
        if e:
            row[k] = ok(e) + tf(e["bar"])
            row[k + " (bar önce)"] = i_son - e["bar"]
            nt = f"{e['ad']} · {'yukarı' if e['dir'] == 1 else 'aşağı'} · seviye {fp(e['seviye'])}"
            if e["hedef"] is not None:
                nt += f" · hedef {fp(e['hedef'])}"
            if e["hacim"]:
                nt += " · Hacim ✓"
            if e["rec"] is not None and e["rec"].get("sonuc"):
                nt += " · " + e["rec"]["sonuc"]
            notlar[k] = nt
        else:
            row[k], row[k + " (bar önce)"] = "", None

    # son barda (bar önce ≤ L0) yukarı / aşağı kırılanlar, virgülle birleşik
    up, dn = [], []
    for e in m.kir:
        if i_son - e["bar"] <= L0:
            (up if e["dir"] == 1 else dn).append(e["ad"])
    row[SON_UP] = ", ".join(dict.fromkeys(up))
    row[SON_DN] = ", ".join(dict.fromkeys(dn))

    olaylar = [{
        "Hisse": hisse, "Periyot": m.aralik, "Kapanış": c, "Yapı türü": kategori(e["ad"]), "Yapı": e["ad"],
        "Yön": "▲ Yukarı" if e["dir"] == 1 else "▼ Aşağı", "Tarih": tf(e["bar"]), "Bar önce": i_son - e["bar"],
        "Seviye": e["seviye"], "Hedef": e["hedef"], "Hacim ✓": "✓" if e["hacim"] else "",
    } for e in m.kir if i_son - e["bar"] <= K]
    return row, olaylar, notlar


def tara(hisseler, K=None, L0=None):
    K = TARAMA_YENI_BAR if K is None else K
    L0 = LISTE_BAR if L0 is None else L0
    liste = list(hisseler)[:MAX_HISSE] if MAX_HISSE else list(hisseler)
    _SON["mum"] = None
    satirlar, olaylar, notlar, hatalar = [], [], [], []
    for h in tqdm(liste, desc=f"Tarama · {ARALIK}"):
        try:
            df, ar = hisse_cek_oto(h)
            m = analiz(df, f"BIST:{h}", ar, AYAR)
            if _SON["mum"] is None or m.idx[-1] > _SON["mum"]:
                _SON["mum"] = m.idx[-1]
            r, o, nt = ozet_satir(m, h, K, L0)
            satirlar.append(r)
            olaylar += o
            notlar.append(nt)
        except Exception as e:
            hatalar.append((h, str(e)[:200]))
        time.sleep(BEKLEME)
    df = pd.DataFrame(satirlar).reindex(columns=SUTUNLAR)
    for k in KATEGORILER:
        df[k + " (bar önce)"] = df[k + " (bar önce)"].astype("Int64")
    olay_df = pd.DataFrame(olaylar)
    if len(olay_df):
        olay_df = olay_df.sort_values(["Bar önce", "Hisse"]).reset_index(drop=True)
    return df, olay_df, notlar, hatalar


def tarama_raporu(df, hatalar):
    if df.empty:
        print("Sonuç yok.")
        return
    print(f"\n{len(df)} hisse tarandı · periyot {ARALIK} · en yeni mum: {_son_mum_txt()}"
          f" · hata/veri yok: {len(hatalar)}")
    alt = df[df["Periyot"] != ARALIK]
    if len(alt):
        print(f"Alt periyoda inenler ({len(alt)}): " + ", ".join(f"{h}({p})" for h, p in zip(alt["Hisse"], alt["Periyot"])))
    df = df.copy()
    df["Değişim %"] = (df["Değişim %"].astype(float) * 100).round(2)

    def blok(baslik, d):
        print(f"\n▌ {baslik}  ({len(d)})")
        if len(d):
            display(d.reset_index(drop=True))
        else:
            print("  (yok)")

    ek = f"bar önce ≤ {LISTE_BAR}"
    blok(f"▲ YUKARI KIRILANLAR ({ek})", df[df[SON_UP] != ""][["Hisse", "Periyot", "Kapanış", "Değişim %", "Trend", SON_UP]])
    blok(f"▼ AŞAĞI KIRILANLAR ({ek})", df[df[SON_DN] != ""][["Hisse", "Periyot", "Kapanış", "Değişim %", "Trend", SON_DN]])
    blok(f"KIRILIMA YAKIN BEKLEYEN FORMASYONLAR (≤ %{YAKINLIK_YUZDE})",
         df[df["Kırılıma yakınlık %"].le(YAKINLIK_YUZDE)][["Hisse", "Periyot", "Kapanış", "Trend", "Aktif formasyon", "Kırılıma yakınlık %"]]
         .sort_values("Kırılıma yakınlık %"))
    if hatalar:
        print(f"\nVeri alınamayanlar ({len(hatalar)}): " + ", ".join(h for h, _ in hatalar[:40])
              + (" …" if len(hatalar) > 40 else ""))


# ─────────────── Excel ───────────────
def _stil_baslik(ws, fills=None):
    for j, c in enumerate(ws[1], start=1):
        c.font = Font(bold=True, color="FFFFFF")
        c.fill = PatternFill("solid", fgColor=(fills[j - 1] if fills else "1F3864"))
        c.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    ws.row_dimensions[1].height = 34
    ws.freeze_panes = "B2"
    ws.auto_filter.ref = ws.dimensions


def _genislik(ws, df, mn=10, mx=46):
    for j, col in enumerate(df.columns, start=1):
        uz = max([len(str(col)) * 0.85] + [len(str(v)) for v in df[col].head(400).tolist()])
        ws.column_dimensions[get_column_letter(j)].width = max(mn, min(mx, uz + 2))


YESIL, KIRMIZI, SARI = "00704A", "C00000", "FFF2CC"


def excele_yaz(df, olay_df, notlar, hatalar, dosya):
    with pd.ExcelWriter(dosya, engine="openpyxl") as w:
        df.to_excel(w, sheet_name="Tarama", index=False)
        (olay_df if len(olay_df) else pd.DataFrame({"Bilgi": ["Bu pencerede olay yok"]})).to_excel(
            w, sheet_name="Yeni Olaylar", index=False)
        pd.DataFrame(hatalar, columns=["Hisse", "Hata"]).to_excel(w, sheet_name="Hatalar", index=False)
        ac = [
            ("Yapı Avcısı – BIST tarama", f"Hedef periyot: {ARALIK} · en yeni mum: {_son_mum_txt()}"),
            ("Sütun çiftleri", "'<Yapı>' = son kırılımın yönü (▲ yukarı / ▼ aşağı) ve tarihi · '<Yapı> (bar önce)' = o kırılımdan bu yana kaç bar"),
            ("Bar önce = 0", "Kırılım, son kapanmış mumda gerçekleşti. Sütunu filtreleyip 0'ı seçin."),
            ("▲ / ▼ sütunları (en sonda)", f"Bar önce ≤ {LISTE_BAR} olan tüm kırılımların adları, birden fazlaysa virgülle birleşik"),
            ("Hücre notu", "Tarih hücrelerinin üzerine gelince yapının tam adı, seviye, hedef, hacim ve sonuç görünür"),
            ("Yapı türleri", "BOS · CHoCH · OBO-TOBO · Üçlü · İkili · Fincan-Kulp · Üçgen · Takoz · Dikdörtgen · Kanal · Bayrak-Flama · Genişleyen · Wyckoff (AL/SAT)"),
            ("Periyot", f"Hisse hangi periyotta analiz edildiyse o yazılır. Hedef periyotta {MIN_BAR} bar yoksa bir alt periyoda inilir (1M→1W→1D→4h→3h→2h→1h). Alt periyoda inenler turuncu işaretlidir; 'bar önce' ve tarihler o satırın periyoduna göredir."),
            ("OHLCV ve Değişim %", "Son kapanmış mumun açılış, yüksek, düşük, kapanış, hacim değerleri; Değişim % = kapanışın bir önceki kapanışa göre değişimi"),
            ("Hatalar sayfası", "En düşük periyoda (1h) inilmesine rağmen yeterli barı olmayan hisseler ve denenen periyotlardaki bar sayıları"),
            ("Uyarı", "Teknik analiz çıktısıdır, yatırım tavsiyesi değildir."),
        ]
        pd.DataFrame(ac, columns=["Konu", "Açıklama"]).to_excel(w, sheet_name="Açıklama", index=False)

        # ── Tarama sayfası biçimi ──
        ws = w.sheets["Tarama"]
        fills = ["1F3864"] * len(TEMEL)
        for gi, k in enumerate(KATEGORILER):
            fills += ["2F5597" if gi % 2 == 0 else "548235"] * 2
        fills += ["00704A", "C00000"]
        _stil_baslik(ws, fills)
        _genislik(ws, df)
        cols = list(df.columns)
        for r in range(2, ws.max_row + 1):
            ws.cell(r, 1).font = Font(bold=True)
            for nm_, fm_ in (("Açılış", "#,##0.00##"), ("Yüksek", "#,##0.00##"), ("Düşük", "#,##0.00##"),
                             ("Kapanış", "#,##0.00##"), ("Hacim", "#,##0"), ("Değişim %", "0.00%")):
                ws.cell(r, cols.index(nm_) + 1).number_format = fm_
            dv = ws.cell(r, cols.index("Değişim %") + 1)
            if isinstance(dv.value, (int, float)) and dv.value == dv.value:
                dv.font = Font(bold=True, color=YESIL if dv.value > 0 else KIRMIZI if dv.value < 0 else "000000")
            pc = ws.cell(r, cols.index("Periyot") + 1)
            pc.alignment = Alignment(horizontal="center")
            if pc.value != ARALIK:
                pc.fill = PatternFill("solid", fgColor="F8CBAD")
                pc.font = Font(bold=True)
            for j, name in enumerate(cols, start=1):
                c = ws.cell(r, j)
                v = c.value
                if isinstance(v, str) and v[:1] in ("▲", "▼") and name in KATEGORILER:
                    c.font = Font(bold=True, color=YESIL if v[0] == "▲" else KIRMIZI)
                    nt = notlar[r - 2].get(name) if r - 2 < len(notlar) else None
                    if nt:
                        c.comment = Comment(nt, "Yapı Avcısı")
                    bc = ws.cell(r, j + 1)
                    bc.alignment = Alignment(horizontal="center")
                    if bc.value == 0:
                        c.fill = PatternFill("solid", fgColor=SARI)
                        bc.fill = PatternFill("solid", fgColor=SARI)
                        bc.font = Font(bold=True)
            for j, col in ((cols.index(SON_UP) + 1, YESIL), (cols.index(SON_DN) + 1, KIRMIZI)):
                c = ws.cell(r, j)
                c.font = Font(bold=True, color=col)
                c.alignment = Alignment(wrap_text=True, vertical="top")
        for col in (SON_UP, SON_DN):
            ws.column_dimensions[get_column_letter(cols.index(col) + 1)].width = 38

        # ── Yeni Olaylar sayfası ──
        wo = w.sheets["Yeni Olaylar"]
        _stil_baslik(wo)
        if len(olay_df):
            _genislik(wo, olay_df)
            yk = list(olay_df.columns).index("Yön") + 1
            for r in range(2, wo.max_row + 1):
                c = wo.cell(r, yk)
                c.font = Font(bold=True, color=YESIL if str(c.value).startswith("▲") else KIRMIZI)
        for nm in ("Hatalar", "Açıklama"):
            _stil_baslik(w.sheets[nm])
        w.sheets["Açıklama"].column_dimensions["A"].width = 30
        w.sheets["Açıklama"].column_dimensions["B"].width = 120
        w.sheets["Hatalar"].column_dimensions["B"].width = 110


def detay(sembol, borsa="BIST", indir=False):
    """Tek hissenin tam raporu + etkileşimli grafik."""
    df, ar = hisse_cek_oto(sembol, borsa)
    m = analiz(df, f"{borsa}:{sembol}", ar, AYAR)
    rapor(m)
    fig = grafik(m, GRAFIK_BAR, HAFTA_SONU_GIZLE)
    fig.show(config=dict(scrollZoom=True, displaylogo=False))
    if indir:
        ad = f"yapi_avcisi_{sembol}_{m.aralik}"
        fig.write_html(ad + ".html", include_plotlyjs="cdn", config=dict(scrollZoom=True, displaylogo=False))
        tablo_df(m).to_excel(ad + ".xlsx", index=False)
        try:
            from google.colab import files
            files.download(ad + ".html")
            files.download(ad + ".xlsx")
        except ImportError:
            print("Kaydedildi:", ad + ".html", ad + ".xlsx")
    return m

#Zaman kontrolü için
def saat_kontrol(sembol="THYAO", aralik="1h", borsa="BIST", adet=10):
    """Saat dilimi kontrolü: BIST 1h barlarının gün içindeki ilk barı 10:00 görünmelidir."""
    df = hisse_cek(sembol, borsa, aralik)
    print(f"{borsa}:{sembol} {aralik} · SAAT_DILIMI = {globals().get('SAAT_DILIMI')!r}")
    print(df.tail(adet)[["open", "close"]])


In [7]:
# saat_kontrol("THYAO", "1M")

## 7 · Taramayı çalıştır
(sayfalar: Tarama · Yeni Olaylar · Hatalar · Açıklama).

In [8]:
tarama_df, olay_df, notlar, tarama_hata = tara(Hisseler)
tarama_raporu(tarama_df, tarama_hata)

# 'Son mum' sütunundaki en son (maksimum) tarihi bulalım
# Tarih formatı dd-mm-yy veya gg.aa.yy şeklinde olabileceğinden noktaları tireye çeviriyoruz
son_mum_serisi = pd.to_datetime(tarama_df['Son mum'], format='%d-%m-%y', errors='coerce')
if son_mum_serisi.isnull().all():
    # Alternatif format denemesi
    son_mum_serisi = pd.to_datetime(tarama_df['Son mum'], format='%d.%m.%y', errors='coerce')

max_tarih = son_mum_serisi.max()
Tarih = max_tarih.strftime('%d-%m-%y') if not pd.isnull(max_tarih) else 'BilinmeyenTarih'

# Dosya adını yeni şablona göre belirleyelim
dosya = f"FormasyonTarama_{dosya}_{Tarih}.xlsx"

print(f"Belirlenen Tarih: {Tarih}")
print(f"Kaydedilecek Dosya Adı: {dosya}")

# Excel dosyasını yazalım
excele_yaz(tarama_df, olay_df, notlar, tarama_hata, dosya)

try:
    from google.colab import files
    files.download(dosya)
except ImportError:
    print("Kaydedildi:", dosya)



Tarama · 1D:   0%|          | 0/626 [00:00<?, ?it/s]

ERROR:tvDatafeed.main:WebSocket receive error for BIST:ERSU: Connection to remote host was lost.



626 hisse tarandı · periyot 1D · en yeni mum: 06-11-2026 18:00 · hata/veri yok: 0
Alt periyoda inenler (23): AAGYO(4h), ALBTN(4h), BETAE(4h), BKRGY(3h), CITAS(4h), EKDMR(4h), EKIM(4h), ENPRA(4h), GOLDA(4h), INTET(3h), ISVEA(4h), KARCL(4h), KPEKS(4h), MASFN(4h), METEN(4h), NETGL(2h), ORZAX(4h), QUICK(4h), SARAE(4h), SOHOE(4h), SSAAT(4h), TKNKA(4h), VEYAS(4h)

▌ ▲ YUKARI KIRILANLAR (bar önce ≤ 0)  (18)


,Hisse,Periyot,Kapanış,Değişim %,Trend,▲ Yukarı kırılanlar
0,ADGYO,1D,48.80,8.01,Yükseliş ▲,CHoCH
1,AZTEK,1D,2.85,2.52,Düşüş ▼,Alçalan Takoz
2,BALAT,1D,55.50,-0.89,Yükseliş ▲,Simetrik Üçgen
3,BIOEN,1D,12.33,9.99,Yükseliş ▲,"CHoCH, İkili Dip"
4,BURCE,1D,20.20,-3.63,Düşüş ▼,Alçalan Takoz
5,CVKMD,1D,14.54,7.62,Yükseliş ▲,CHoCH
6,DGNMO,1D,5.41,9.96,Yükseliş ▲,CHoCH
7,FROTO,1D,73.15,7.81,Yükseliş ▲,CHoCH
8,IDGYO,1D,1.71,1.79,Düşüş ▼,Alçalan Takoz
9,ISBIR,1D,64.00,3.23,Yükseliş ▲,CHoCH



▌ ▼ AŞAĞI KIRILANLAR (bar önce ≤ 0)  (2)


,Hisse,Periyot,Kapanış,Değişim %,Trend,▼ Aşağı kırılanlar
0,PAGYO,1D,105.4,-4.18,Düşüş ▼,BOS
1,TRCAS,1D,39.0,-1.27,Düşüş ▼,BOS



▌ KIRILIMA YAKIN BEKLEYEN FORMASYONLAR (≤ %2.0)  (4)


,Hisse,Periyot,Kapanış,Trend,Aktif formasyon,Kırılıma yakınlık %
0,ORZAX,4h,66.90,Yükseliş ▲,İkili Dip ↑,0.67
1,ISBTR,1D,379600.00,Düşüş ▼,Düşen Kanal,1.80
2,ULUFA,1D,1.62,Yükseliş ▲,İkili Dip ↑,1.85
3,DMLKT,1D,6.98,Düşüş ▼,Fincan-Kulp ↑,1.86


Belirlenen Tarih: 06-11-26
Kaydedilecek Dosya Adı: FormasyonTarama_Gunluk_06-11-26.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 8 · Hisse detayı (etkileşimli grafik + indirme)

In [ ]:
# Taramada ilginizi çeken hisseleri etkileşimli grafikle inceleyin
DETAY_HISSELER = ["THYAO", "GARAN","SAYAS"]
INDIR = True        # HTML grafik + Excel tablo indirilsin mi

for h in DETAY_HISSELER:
    try:
        detay(h, indir=INDIR)
    except Exception as e:
        print(f"[HATA] {h} → {e}")


## 9 · Hisse + yapı → grafik
Sunucudaki günlük raporda gördüğünüz hisseyi burada çizdirin. Önce 4. ve 6. hücreler (motor, grafik) ile 8. hücre (ayarlar) çalışmış olmalı; 12. hücre (`hisse_cek`) de gerekli. Yalnızca seçtiğiniz yapının en güncel kırılımı çizilir.

In [9]:
# ═══════════════ HİSSE + YAPI → GRAFİK ═══════════════
# Sunucudaki rapordan / Excel'den ilgilendiğiniz hisseyi ve yapıyı seçip burada çizdirin.
#   ciz("ACSEL", "CHoCH")                      → son CHoCH'u çizer
#   ciz("BINHO", "Kanal")                      → son kanal kırılımını (formasyonu) çizer
#   ciz("THYAO", "BOS, Yükselen Kanal")        → rapor hücresindeki gibi virgüllü verilebilir
#   ciz("TAVHL", "bekleyen")                   → kırılım bekleyen formasyonları çizer
#   ciz("AAGYO", "CHoCH", periyot="4h")        → raporda Periyot 4h ise (alt periyoda inmişse)
#   ciz("ACSEL")                               → tüm yapılar (detay gibi) + son kırılımların listesi
import copy


def _n(s):
    """Türkçe karakterlerden bağımsız karşılaştırma: 'İkili' = 'ikili' = 'IKILI', 'Üçgen' = 'ucgen'."""
    t = str(s).replace("İ", "i").replace("I", "ı").lower()
    for a, b in (("ı", "i"), ("ç", "c"), ("ğ", "g"), ("ö", "o"), ("ş", "s"), ("ü", "u")):
        t = t.replace(a, b)
    return t.strip()


def _olay_sec(m, yapi, son_n=1):
    """m.kir (tüm kırılımlar) içinden yapıya uyan EN GÜNCEL olay(lar). Eşleşme: tam ad → yapı türü → içerir."""
    q = _n(yapi)
    gruplar = (
        [e for e in m.kir if _n(e["ad"]) == q],                    # "OBO", "BOS", "Yükselen Kanal"
        [e for e in m.kir if _n(kategori(e["ad"])) == q],          # "Kanal", "Üçgen", "Wyckoff" (Excel sütun adları)
        [e for e in m.kir if q in _n(e["ad"])],                    # "yukselen", "takoz" ...
    )
    for g in gruplar:
        if g:
            return sorted(g, key=lambda e: e["bar"])[-son_n:]
    return []


def _bekleyenler(m, yapi=None):
    """Kırılım bekleyen formasyonlar; yapi verilirse adına / türüne göre süzülür."""
    liste = [r for r in m.hist if r["durum"] == "Kırılım bekliyor"]
    if yapi and _n(yapi) not in ("bekleyen", "bekliyor", "yakin"):
        q = _n(yapi)
        liste = [r for r in liste if q in _n(r["ad"]) or _n(kategori(r["ad"].split(" ↑")[0].split(" ↓")[0])) == q]
    return liste


def _rec_bas(r):
    if r["tur"] == "Dönüş":
        return min(b for b, _ in r["noktalar"])
    return min(r["ust"][0], r["alt"][0])


def _mevcut_yapilar(m):
    """Bu hissede bulunan son kırılımlar (yapı türü başına) ve bekleyen formasyonlar."""
    fmt = "%d-%m-%y" if m.aralik in ("1D", "1W", "1M") else "%d-%m-%y %H:%M"
    son = {}
    for e in m.kir:
        son[kategori(e["ad"])] = e
    print("  Son kırılımlar (yapı türü → yön · tarih · kaç bar önce):")
    if not son:
        print("   (yok)")
    for k, e in sorted(son.items(), key=lambda kv: -kv[1]["bar"]):
        print(f"   {k:<16} {'▲' if e['dir'] == 1 else '▼'} {m.idx[e['bar']].strftime(fmt)} · {m.n - 1 - e['bar']} bar önce · {e['ad']}")
    bek = _bekleyenler(m)
    if bek:
        print("  Kırılım bekleyen:", ", ".join(r["ad"] for r in bek))


def ciz(hisse, yapi=None, periyot=None, borsa="BIST", son_n=1, bar=None, indir=False):
    """Hisse + yapı ver → yalnızca o yapıyı (en günceli) çizen etkileşimli grafik.

    yapi    : "CHoCH" | "BOS" | formasyon adı ("Yükselen Kanal") | yapı türü ("Kanal", "Üçgen", "Takoz",
              "OBO-TOBO", "İkili Tepe-Dip", "Fincan-Kulp", "Bayrak-Flama", "Wyckoff" ...) | "bekleyen"
              Virgülle birden fazla verilebilir. None → tüm yapılar (detay gibi).
    periyot : None → ARALIK (bar yetersizse alt periyoda iner). "4h" vb. verilirse doğrudan o periyot.
    son_n   : aynı türden kaç son kırılım çizilsin (varsayılan 1).
    bar     : grafikte gösterilecek son bar sayısı (None → yapıya göre otomatik yakınlaştırma).
    """
    if periyot:
        df = hisse_cek(hisse, borsa, periyot)
        ar = periyot
    else:
        df, ar = hisse_cek_oto(hisse, borsa)
    m = analiz(df, f"{borsa}:{hisse}", ar, AYAR)
    fmt = "%d-%m-%y" if ar in ("1D", "1W", "1M") else "%d-%m-%y %H:%M"
    print(f"{borsa}:{hisse} · {ar} · son mum {m.idx[-1].strftime(fmt)} · kapanış {fp(m.c[-1])}")

    if not yapi:
        _mevcut_yapilar(m)
        fig = grafik(m, bar or GRAFIK_BAR, HAFTA_SONU_GIZLE)
        baslik = "tüm yapılar"
    else:
        olaylar, bekleyen = [], []
        for tok in [t.strip() for t in str(yapi).split(",") if t.strip()]:
            if _n(tok) in ("bekleyen", "bekliyor", "yakin"):
                bekleyen += _bekleyenler(m)
                continue
            ev = _olay_sec(m, tok, son_n)
            if ev:
                olaylar += ev
            else:                                               # kırılım yok → aynı adla bekleyen formasyon var mı
                bekleyen += _bekleyenler(m, tok)
        olaylar = [e for i, e in enumerate(olaylar) if all(e is not x for x in olaylar[:i])]
        if not olaylar and not bekleyen:
            print(f"  ⚠ '{yapi}' için kırılım / bekleyen formasyon bulunamadı.")
            _mevcut_yapilar(m)
            return

        m2 = copy.copy(m)
        m2.zz, m2.swLabels, m2.bosList, m2.hist, m2.wyHist = [], [], [], [], []
        bas = m.n - 1
        for e in olaylar:
            if e["ad"] in ("BOS", "CHoCH"):
                for t in m.bosList:
                    if t[0] == e["bar"] and t[2] == e["ad"]:
                        m2.bosList.append(t)
                        bas = min(bas, t[1])
            elif e["ad"].startswith("Wyckoff"):
                if all(e["rec"] is not x for x in m2.wyHist):
                    m2.wyHist.append(e["rec"])
                    bas = min(bas, e["rec"]["sc_bar"])
            elif all(e["rec"] is not x for x in m2.hist):
                m2.hist.append(e["rec"])
                bas = min(bas, _rec_bas(e["rec"]))
        for r in bekleyen:
            if all(r is not x for x in m2.hist):
                m2.hist.append(r)
                bas = min(bas, _rec_bas(r))

        for e in olaylar:
            print(f"  • {e['ad']} {'▲' if e['dir'] == 1 else '▼'} · {m.idx[e['bar']].strftime(fmt)} · {m.n - 1 - e['bar']} bar önce"
                  f" · seviye {fp(e['seviye'])}" + (f" · hedef {fp(e['hedef'])}" if e["hedef"] is not None else "")
                  + (" · Hacim ✓" if e["hacim"] else "")
                  + (f" · {e['rec']['sonuc']}" if e.get("rec") is not None and e["rec"].get("sonuc") else ""))
        for r in bekleyen:
            print(f"  • {r['ad']} · kırılım bekliyor")

        goster = bar or max(60, min(m.n, m.n - max(0, bas - 25) + 5))     # yapıdan ~25 bar öncesinden itibaren
        fig = grafik(m2, goster, HAFTA_SONU_GIZLE)
        baslik = ", ".join([e["ad"] + (" ↑" if e["dir"] == 1 else " ↓") for e in olaylar] + [r["ad"] + " (bekliyor)" for r in bekleyen])
    fig.update_layout(title=f"{hisse} · {ar} · {baslik} · son mum {m.idx[-1].strftime(fmt)} · kapanış {fp(m.c[-1])}")
    fig.show(config=dict(scrollZoom=True, displaylogo=False))
    if indir:
        ad = f"yapi_{hisse}_{ar}"
        fig.write_html(ad + ".html", include_plotlyjs="cdn", config=dict(scrollZoom=True, displaylogo=False))
        try:
            from google.colab import files
            files.download(ad + ".html")
        except ImportError:
            print("Kaydedildi:", ad + ".html")


### Örnek çağrılar

In [11]:
# Sunucu raporundan / Excel'den gördüğünüz hisse ve yapıyı yazın.
# Raporda Periyot 4h ise (alt periyoda inmişse) periyot="4h" ekleyin; yoksa ARALIK kullanılır.
ciz("TAVHL", "TOBO")

# ciz("BINHO", "Kanal")                    # formasyon (yapı türü: Kanal, Üçgen, Takoz, OBO-TOBO, İkili Tepe-Dip, ...)
# ciz("THYAO", "BOS, Yükselen Kanal")      # rapor hücresindeki gibi virgüllü
# ciz("TAVHL", "bekleyen")                 # kırılım bekleyen formasyon
# ciz("AAGYO", "CHoCH", periyot="4h")
# ciz("ACSEL")                             # tüm yapılar + hissedeki son kırılımların listesi


BIST:TAVHL · 1D · son mum 06-11-26 · kapanış 294.50
  • TOBO ▲ · 06-11-26 · 0 bar önce · seviye 293.30 · hedef 332.60
